In [ ]:
#!pip install pandas numpy missingno openpyxl unidecode

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re
import missingno as msno

In [ ]:
import warnings
# Remove excel styles warnings that can cause issues with openpyxl when reading files with complex formatting
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


# 1. Vital signs #

### 2022 to 2025

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re

# -------CONSTANTS---------
DATA_PATH = "../Datanad/subset_data"

YEAR_MIN = 2022
YEAR_MAX = 2025

OUTPUT_PATH = "Datasets"
OUTPUT_FILE = f"df_param_vit_{YEAR_MIN}_{YEAR_MAX}.csv"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Step 1: File screening ---
all_files = glob.glob(os.path.join(DATA_PATH, "*.xlsx"))

files = [
    f for f in all_files
    if "CGJ" in os.path.basename(f).upper()
       and "052C" in os.path.basename(f).upper()
       and not os.path.basename(f).startswith("~$")
]
files = sorted(files)
print(f"--- CHECK : {len(files)} files found ---")

# --- Step 2: Load ---
dfs = []
for file in files:
    try:
        df = pd.read_excel(file, header=3)
        df.columns = df.columns.astype(str).str.strip()
        df['source_file'] = os.path.basename(file)
        dfs.append(df)
        print(f" ✅ loaded : {os.path.basename(file)}")
    except Exception as e:
        print(f" ❌ Error on {os.path.basename(file)} : {e}")

if not dfs:
    print("⚠️ Aucun fichier chargé — arrêt.")
    exit()

df_concat = pd.concat(dfs, ignore_index=True)
print(f"Total rows loaded: {len(df_concat)}")

# --- Step 3: Cleaning functions ---
def clean_numeric(x):
    if pd.isna(x) or str(x).lower() in ["na", "n/a", "", "nan", "none"]:
        return np.nan
    s = str(x).strip().lower().replace(',', '.')
    s = re.sub(r'[^0-9.]', '', s)
    try:
        return float(s) if s != "" else np.nan
    except:
        return np.nan

def merge_columns_regex(df, regex_candidates, new_col_name):
    series = None
    for pattern in regex_candidates:
        matching_cols = [col for col in df.columns if re.search(pattern, str(col), re.IGNORECASE)]
        for col in matching_cols:
            if series is None:
                series = df[col]
            else:
                series = series.combine_first(df[col])
    if series is not None:
        df[new_col_name] = series
    return df

# --- Step 4: Fusion ---
df_merged = df_concat.copy()

# 4a. Colonnes par position
df_merged["nda"]              = df_merged.iloc[:, 2]
df_merged["date_adm_vitals"]  = df_merged.iloc[:, 3]


# 4b. Colonnes par regex
columns_mapping = {
    "urine_dipstick":  [r"Bandelette\s*Urinaire"],
    "sbp":             [r"TA\s*Max\s*Bras", r"TA\s*Syst"],
    "dbp":             [r"TA\s*Min\s*Bras", r"TA\s*Diast"],
    "hr":              [r"Fr[ée]quence\s*Cardiaque"],
    "temp":            [r"Temp[ée]rature"],
    "sat":             [r"Saturation\s*P[ée]riph[ée]rique"],
    "rr":              [r"Fr[ée]quence\s*Respiratoire"],
    "o2_flow":         [r"D[ée]bit.*Oxyg[èe]ne"],
    "cap_blood_sugar": [r"Glyc[ée]mie\s*Digitale"],
    "hemocue":         [r"Hemocue"],
    "gcs":             [r"Glasgow", r"Score\s*de\s*Glasgow"],
    "pain":            [r"EN\."],
    "breathalyzer":    [r"Ethylotest"],
    "pupil_right":     [r"Pupillaire\s*Droit"],
    "pupil_left":      [r"Pupillaire\s*Gauche"],
}

for new_col, regex_list in columns_mapping.items():
    df_merged = merge_columns_regex(df_merged, regex_list, new_col)

# --- Step 5: Sélection des colonnes finales ---
final_cols = ["nda", "date_adm_vitals"] + \
             list(columns_mapping.keys()) + ["source_file"]
df_final = df_merged[[col for col in final_cols if col in df_merged.columns]].copy()

# --- Step 6: Cleaning ---

# 1. Dates
df_final["date_adm_vitals"]  = pd.to_datetime(df_final["date_adm_vitals"],  errors="coerce")


# 2. Diagnostic années
print("=== Distribution des années dans date_adm_vitals ===")
print(df_final["date_adm_vitals"].dt.year.value_counts().sort_index())
print(f"\ndate_adm_vitals NaT : {df_final['date_adm_vitals'].isna().sum()}")


# 3. Year filter
df_final = df_final[
    df_final["date_adm_vitals"].dt.year.between(YEAR_MIN, YEAR_MAX)
]
print(f"Rows after year filter ({YEAR_MIN}-{YEAR_MAX}): {len(df_final)}")

# 4. Numerical columns
quant_columns = ["sbp", "dbp", "hr", "temp", "sat", "rr", "cap_blood_sugar",
                 "hemocue", "gcs", "pain", "breathalyzer", "pupil_right", "pupil_left", "o2_flow"]

for col in quant_columns:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(clean_numeric)

# 5. Urine dipstick
if "urine_dipstick" in df_final.columns:
    df_final["urine_dipstick"] = df_final["urine_dipstick"].astype(str).replace(
        ['nan', 'None', 'NaN', ''], np.nan
    )

# 6. Hospital
df_final["hospital"] = np.where(
    df_final["source_file"].str.contains("SA", case=False, na=False), "SA", "PEL"
)

# --- Step 7: Sort and export ---
df_final = df_final.sort_values(by=["nda", "date_adm_vitals"]).reset_index(drop=True)
df_final.to_csv(os.path.join(OUTPUT_PATH, OUTPUT_FILE), index=False)

print(f"\n🚀 Done! {len(df_final)} rows.")
print(f"Columns: {df_final.columns.tolist()}")

# --- Checks ---
print("\n=== Vitals coverage ===")
quant_cols = [c for c in quant_columns if c in df_final.columns]
print(df_final[quant_cols].notna().sum().sort_values(ascending=False))

print("\n=== Urine dipstick check ===")
if "urine_dipstick" in df_final.columns:
    valides = df_final["urine_dipstick"].dropna()
    print(f"Filled rows: {len(valides)}")
    print(valides.value_counts().head(5))

print("\n=== Pupils check ===")
if all(c in df_final.columns for c in ["pupil_right", "pupil_left"]):
    print(df_final[["pupil_right", "pupil_left"]].describe())

In [ ]:
# #===============================================
# # CODE FOR EXCEL FILES WITH PIVOT
# # ==============================================
#
# #-------CONSTANT---------
# DATA_PATH = "../Datanad/subset_data"
#
#
# YEAR_MIN = 2022
# YEAR_MAX = 2025
#
# OUTPUT_PATH = "Datasets"
# OUTPUT_FILE = f"df_param_vit_{YEAR_MIN}_{YEAR_MAX}.csv"
# os.makedirs(OUTPUT_PATH, exist_ok=True)
#
#
# # --- Step 1: File screening ---
#
# all_files = glob.glob(os.path.join(DATA_PATH, "*.xlsx"))
#
#
# files = [
#     f for f in all_files
#     if "CGJ" in os.path.basename(f).upper()
#        and "052C" in os.path.basename(f).upper()
#        and not os.path.basename(f).startswith("~$")
# ]
# files = sorted(files)
# print(f"--- CHECK : {len(files)} files found ---")
#
# # --- Step 2: Load with explicit column names (no header) ---
# COL_NAMES = ["code_a", "nda", "date_adm", "vital_name", "vital_value",
#              #"unit_code", "date_hour_vitals", "year"
#              ]
#
# dfs = []
# for file in files:
#     try:
#         df = pd.read_excel(file, header=None, names=COL_NAMES)
#         df['source_file'] = os.path.basename(file)
#         dfs.append(df)
#         print(f" ✅ loaded : {os.path.basename(file)}")
#     except Exception as e:
#         print(f" ❌ Error on {os.path.basename(file)} : {e}")
#
# df_concat = pd.concat(dfs, ignore_index=True)
# print(f"Total rows loaded: {len(df_concat)}")
#
# # --- Step 3: Cleaning functions ---
#
# def clean_numeric(x):
#     if pd.isna(x) or str(x).lower() in ["na", "n/a", "", "nan", "none"]:
#         return np.nan
#     s = str(x).strip().lower().replace(',', '.')
#     s = re.sub(r'[^0-9.]', '', s)
#     try:
#         return float(s) if s != "" else np.nan
#     except:
#         return np.nan
#
# # Vital name → standard column name mapping (regex)
# vitals_mapping = {
#     "sbp":           [r"TA\s*[-\s]*[Mm]ax", r"TA\s*Syst"],
#     "dbp":           [r"TA\s*[-\s]*[Mm]in", r"TA\s*Diast"],
#     "hr":            [r"Fr[ée]quence\s*Cardiaque", r"FC\s*concat"],
#     "temp":          [r"Temp[ée]ratu"],
#     "sat":           [r"Sp[O0]2\s*conc", r"Saturation"],
#     "rr":            [r"Fr[ée]quence\s*[Rr]espiratoire", r"Frequence\s*[Rr]"],
#     "cap_blood_sugar":[r"Glyc[ée]mie"],
#     "gcs":           [r"Glasgow", r"Score\s*de\s*Glasgow"],
#     "pain":          [r"EN\.", r"Echelle\s*de\s*douleur", r"EN\s*concat"],
#     "urine_dipstick": [r"Bandelette\s*Urinaire", r"BU"],
#     "hemocue":       [r"Hemocue"],
#     "breathalyzer":  [r"Ethylotest", r"OH_expi", r"Alcool"],
#     "pupil_right":   [r"Pupill.*[Dd]roit"],
#     "pupil_left":    [r"Pupill.*[Gg]auche"],
#     "o2_flow":       [r"D[ée]bit.*[Oo]xyg"],
# }
#
# def match_vital(vital_name_str):
#     """Maps a raw vital name string to a standard column name using regex."""
#     if pd.isna(vital_name_str):
#         return None
#     s = str(vital_name_str).strip()
#     for std_name, patterns in vitals_mapping.items():
#         for pattern in patterns:
#             if re.search(pattern, s, re.IGNORECASE):
#                 return std_name
#     return None  # unrecognized vital → will be dropped
#
# # --- Step 4: Map vital names ---
# df_concat["vital_std"] = df_concat["vital_name"].apply(match_vital)
#
# # Quick check: how many vitals were recognized?
# print("\n=== Vital name mapping check ===")
# print(df_concat.groupby("vital_name")["vital_std"].first().to_string())
#
# # Drop rows where vital not recognized (Date et Heure rows, unknowns...)
# df_known = df_concat[df_concat["vital_std"].notna()].copy()
# print(f"\nRows after dropping unknown vitals: {len(df_known)}")
#
# # --- Step 5: Clean dates and values ---
# #df_known["date_hour_vitals"] = pd.to_datetime(df_known["date_hour_vitals"], errors="coerce")
# df_known["date_adm"] = pd.to_datetime(df_known["date_adm"], errors="coerce")
# df_known["vital_value"] = df_known["vital_value"].apply(clean_numeric)
#
# # --- Step 6: Pivot long → wide ---
# # One row per (nda, date_hour_vitals), one column per vital
# df_pivot = df_known.pivot_table(
#     index=["nda", "date_adm",
#            #"date_hour_vitals", "unit_code",
#            "source_file"],
#     columns="vital_std",
#     values="vital_value",
#     aggfunc="first"  # if duplicates, keep first
# ).reset_index()
#
# # Flatten column names
# df_pivot.columns.name = None
#
# # --- Step 7: Hospital identification ---
# df_pivot["hospital"] = np.where(df_pivot["source_file"].str.contains("SA", case=False, na=False), "SA", "PEL")
#
# # --- Step 8: Sort and export ---
# df_final = df_pivot.sort_values(by=["nda",
#                                     #"date_hour_vitals"
#                                     "date_adm"
#                                     ]).reset_index(drop=True)
#
# # SAVE
# df_final.to_csv(OUTPUT_FILE, index=False)
#
# print(f"\n🚀 Done! {len(df_final)} rows (one per patient/timestamp).")
# print(f"Columns: {df_final.columns.tolist()}")
#
# # --- Checks ---
# print("\n=== Vitals coverage ===")
# quant_cols = [c for c in vitals_mapping.keys() if c in df_final.columns]
# print(df_final[quant_cols].notna().sum().sort_values(ascending=False))

## 2022

In [ ]:
# import os
# import glob
# import pandas as pd
# import numpy as np
# import re
# import warnings
#
#
# # --- Step 1 : file screening ---
# DATA_PATH = "../Datanad/subset_2022_PEL"
# tous_fichiers = glob.glob(os.path.join(DATA_PATH, "*.xlsx"))
#
# # Surgical selection of CGJ 052c files
# files = [
#     f for f in tous_fichiers
#     if "CGJ" in os.path.basename(f).upper()
#        and "052C" in os.path.basename(f).upper()
#        and not os.path.basename(f).startswith("~$")
# ]
# files = sorted(files)
#
# print(f"--- CHECK : {len(files)} files found ---")
#
# dfs = []
# for file in files:
#     try:
#         # header=3 to read from line 4, which contains the actual column names
#         df = pd.read_excel(file, header=3)
#         # On nettoie les noms de colonnes des espaces invisibles
#         df.columns = df.columns.astype(str).str.strip()
#         # On ajoute le nom du fichier pour la traçabilité
#         df['source_file'] = os.path.basename(file)
#         dfs.append(df)
#         print(f" ✅ loaded : {os.path.basename(file)}")
#     except Exception as e:
#         print(f" ❌ Error on {os.path.basename(file)} : {e}")
#
# df_concat = pd.concat(dfs, ignore_index=True)
#
# # --- Step 2: cleaning and merging functions ---
#
# def clean_numeric(x):
#     """Transforms text in nombers (ex: '120 bpm' -> 120.0)"""
#     if pd.isna(x) or str(x).lower() in ["na", "n/a", "", "nan", "none"]:
#         return np.nan
#     s = str(x).strip().lower()
#     s = s.replace(',', '.')
#     # keep only digits and dots (for decimals)
#     s = re.sub(r'[^0-9.]', '', s)
#     try:
#         return float(s) if s != "" else np.nan
#     except:
#         return np.nan
#
# def merge_columns_regex(df, regex_candidates, new_col_name):
#     """Raw merging based on regex patterns, without any assumption on the column names"""
#     series = None
#     for pattern in regex_candidates:
#         matching_cols = [col for col in df.columns if re.search(pattern, str(col), re.IGNORECASE)]
#         for col in matching_cols:
#             if series is None:
#                 series = df[col]
#             else:
#                 # fills the missing values in series with the values from df[col]
#                 series = series.combine_first(df[col])
#
#     if series is not None:
#         df[new_col_name] = series
#     return df
#
# # --- Step 3 : Columns  mapping ---
#
# columns_mapping = {
#     "nda": [r"NDA", r"N°\s*DA", r"Unnamed:\s*2"],
#     "date_hour_adm_vitals_file": [r"Unnamed:\s*3"],
#     "date_hour_vitals": [r"Date\s*/\s*heure", r"^Date\s*et\s*Heure$"],
#     "urine_dipstick": [r"Bandelette", r"BU"],
#     "sbp": [r"TA\s*[-\s]*Max", r"TA\s*Syst"],
#     "dbp": [r"TA\s*[-\s]*Min", r"TA\s*Diast"],
#     "hr": [r"Fr[ée]quence\s*Cardiaque", r"FC\s*concat"],
#     "temp": [r"Temp[ée]rature"],
#     "sat": [r"Sp02\s*concat", r"Saturation\s*P[ée]riph[ée]rique", r"Saturation\s*En\s*O2"],
#     "rr": [r"Fr[ée]quence\s*Respiratoire"],
#     "o2_flow": [r"D[ée]bit.*Oxyg[èe]ne"],
#     "cap_blood_sugar": [r"Glyc[ée]mie\s*Digitale", r"Glyc[ée]mie\s*digital"],
#     "hemocue": [r"Hemocue"],
#     "gcs": [r"Glasgow", r"Score\s*de\s+Glasgow"],
#     "pain": [r"EN\.", r"Echelle\s*de\s*douleur", r"EN\s*concat"],
#     "breathalyzer": [r"Ethylotest", r"OH_expi", r"Alcoolémie"],
#     "pupil_right": [r"Pupillaire\s*Droit", r"Pupille\s*Droit"],
#     "pupil_left": [r"Pupillaire\s*Gauche", r"Pupille\s*Gauche"]
# }
#
# # --- Step 4 : Fusion run ---
#
# df_merged = df_concat.copy()
# for new_col, regex_list in columns_mapping.items():
#     df_merged = merge_columns_regex(df_merged, regex_list, new_col)
#
# # final column selection (we keep the merged columns and the source_file for traceability)
# final_cols = list(columns_mapping.keys()) + ["source_file"]
# df_final = df_merged[[col for col in final_cols if col in df_merged.columns]].copy()
#
# # --- Step 5: Cleaning (Uniformization of Excel/Text formats) ---
#
# # Numerical column list
# quant_columns = ["sbp", "dbp", "hr", "temp", "sat", "rr", "cap_blood_sugar", "hemocue",
#                  "gcs", "pain", "breathalyzer", "pupil_right", "pupil_left", "o2_flow"]
#
# # 1. cleaning numerical columns (removing units, converting to float, handling 'nan' as np.nan)
# for col in quant_columns:
#     if col in df_final.columns:
#         df_final[col] = df_final[col].apply(clean_numeric)
#
# # 2. Cleaning the urine dipstick column) to have consistent NaN values
# if "urine_dipstick" in df_final.columns:
#     df_final["urine_dipstick"] = df_final["urine_dipstick"].astype(str).replace(['nan', 'None', 'NaN', 'nan', ''], np.nan)
#
# # 3. Date conversion
# if "date_hour_vitals" in df_final.columns:
#     df_final["date_hour_vitals"] = pd.to_datetime(df_final["date_hour_vitals"], errors="coerce")
#
# # 4. Hospital identification based on the source file name (SA or PEL)
# df_final['hospital'] = np.where(df_final['source_file'].str.contains('SA', case=False, na=False), 'SA', 'PEL')
#
# # --- Step 6 : Export and check ---
#
# df_final = df_final.sort_values(by=["nda", "date_hour_vitals"])
# df_final.to_csv("df_param_vit_subset22pel.csv", index=False)
#
# print(f"\n🚀 Done ! {len(df_final)} trated lines.")
# print(f"final columns : {df_final.columns.tolist()}")
#
# # --- LE TEST DE VÉRITÉ ---
# print("\n=== urine disptick checking ===")
# if "urine_dipstick" in df_final.columns:
#     valides = df_final['urine_dipstick'].dropna()
#     print(f"nomber of filled urine dipstick : {len(valides)}")
#     print("Examples :")
#     print(valides.value_counts().head(5))
#
# print("\n=== Pupils checking ===")
# print(df_final[['pupil_right', 'pupil_left']].describe())

# 2. Triage nurse files #

### from 2022 to 2025

In [58]:
#===================================================
# IOA TRIAGE FILES - HARMONIZED PROCESSING (2022-2025)
# New format: long → wide (no skiprows, named columns)
#===================================================
import os
import glob
import re
import pandas as pd
import numpy as np
import unidecode

# ----------------------------
# CONSTANTS
# ----------------------------
DATA_PATH = "../Datanad/subset_data"
YEAR_MIN = 2022
YEAR_MAX = 2025
os.makedirs("Datasets", exist_ok=True)
output_file = f"Datasets/df_ioa_{YEAR_MIN}_{YEAR_MAX}.csv"


# ----------------------------
# 1️⃣ FILE INVENTORY
# ----------------------------
def get_ioa_files(pattern="CGJ 030*"):
    files = glob.glob(os.path.join(DATA_PATH, pattern))
    files = [f for f in files if not os.path.basename(f).startswith("~$")]
    print(f"--- INVENTORY: {len(files)} files detected ---")
    return sorted(files)

# ----------------------------
# 2️⃣ LOAD AND CONCATENATE
# ----------------------------
def load_files(files):
    dfs = []
    for f in files:
        try:
            df_temp = pd.read_excel(f, header=0)
            df_temp.columns = df_temp.columns.astype(str).str.strip()
            df_temp['source_file'] = os.path.basename(f)
            dfs.append(df_temp)
            print(f" ✅ Loaded: {os.path.basename(f)}")
        except Exception as e:
            print(f" ❌ Error: {os.path.basename(f)} : {str(e)[:50]}")
    df = pd.concat(dfs, ignore_index=True)
    print(f"Total rows loaded: {len(df)}")
    return df

# ----------------------------
# 3️⃣ MAP QUESTION LABELS → STANDARD COLUMN NAMES
# ----------------------------
questions_mapping = {
    "triage_raw":              [r"Tri\s*IAO", r"Score\s*de\s*Gravit[ée]", r"Niveau\s*de\s*Triage"],
    "transport":               [r"Mode\s*d.arriv[ée]e", r"Mode\s*de\s*Transport"],
    "atcd_ioa":                [r"Ant[ée]c[ée]dents"],
    "chief_complaint":         [r"Motifs?\s*de\s*recours", r"Motif\s*de\s*consultation"],
    "anam_ioa":                [r"Circonstances", r"Commentaires\s*aux\s*urgences",
                                r"Histoire\s*de\s*la\s*maladie", r"Anamn[èe]se"],
    "ttt_adm_ioa_file":        [r"Traitement\s*administr[ée]", r"Traitement\s*en\s*cours"],
    "admission_summary_ioa":   [r"Synth[èe]se\s*PEC", r"Synth[èe]se\s*Initiale"],
    "evolution_ioa":           [r"Evolution"],
    "date_hour_triage_begin":  [r"Date\s*et\s*heure"],
}

def match_question(label):
    if pd.isna(label):
        return None
    s = str(label).strip()
    for std_name, patterns in questions_mapping.items():
        for pattern in patterns:
            if re.search(pattern, s, re.IGNORECASE):
                return std_name
    return None

# ----------------------------
# 4️⃣ IDENTIFY COLUMN NAMES
# ----------------------------
def find_col(df, patterns):
    for p in patterns:
        for col in df.columns:
            if re.search(p, str(col), re.IGNORECASE):
                return col
    return None

# ----------------------------
# 5️⃣ HARMONIZE TRIAGE SCORE
# ----------------------------
def harmonize_triage(x):
    if pd.isna(x): return np.nan
    s = unidecode.unidecode(str(x)).lower().strip()

    # --- LEVEL 1 ---
    if any(k in s for k in ["reanim", "sans delai", "medecin <1min", "medecin < 1min"]):
        return "1"

    # --- LEVEL 2 ---
    if any(k in s for k in ["tres urgent", "très urgent", "medecin < 20min", "medecin <20min",
                              "infirmiere <10min", "infirmière <10min"]):
        return "2"

    # --- LEVEL 4 (avant niveau 3 pour éviter que "urgent" attrape "peu urgent") ---
    if any(k in s for k in ["peu urgent", "medecin <2h", "medecin < 2h",
                              "medecin <120min", "medecin < 120min", "120min"]):
        return "4"

    # --- LEVEL 5 ---
    if any(k in s for k in ["non urgent", "medecin <3h", "medecin < 3h",
                              "medecin <240min", "medecin < 240min", "240min"]):
        return "5"

    # --- LEVEL 3 (en dernier car "urgent" est présent dans niveau 2 et 4) ---
    if any(k in s for k in ["urgent", "medecin <1h", "medecin < 1h",
                              "medecin <60min", "medecin < 60min",
                              "medecin <90min", "medecin < 90min",
                              "60min", "90min"]):
        return "3"

    return np.nan

# ----------------------------
# 6️⃣ MAIN SCRIPT
# ----------------------------
def main():
    files = get_ioa_files()
    df_raw = load_files(files)

    # --- Identify key columns ---
    col_nda        = find_col(df_raw, [r"NDA", r"N°\s*DA", r"N°\s*dossier"])
    col_question   = find_col(df_raw, [r"Libell[ée]\s*question"])
    col_value      = find_col(df_raw, [r"Libell[ée]\s*court\s*ou\s*long", r"Libell[ée]\s*long"])
    col_date_adm   = find_col(df_raw, [r"Date\s*Entr[ée]e\s*S[ée]jour\s*avec\s*heure", r"Date\s*Entr[ée]e\s*S[ée]jour"])
    col_sex        = find_col(df_raw, [r"Sexe"])
    col_age        = find_col(df_raw, [r"Age"])
    col_year       = find_col(df_raw, [r"Ann[ée]e\s*entr[ée]e"])
    col_uam        = find_col(df_raw, [r"Code\s*UAM"])
    col_date_triage_end = find_col(df_raw, [r"Date\s*cr[ée]ation\s*questionnaire",
                                            r"Date\s*creation\s*questionnaire",
                                            r"Date\s*cr\u00e9ation"   # ← unicode explicite pour é
                                            ])

    print(f"col_date_triage_end → {col_date_triage_end}")
    print(f"patient_cols sera → {[c for c in [col_nda, col_date_adm, col_sex, col_age, col_uam, col_date_triage_end, 'source_file'] if c]}")


    print("\n=== Column mapping detected ===")
    for name, col in [("nda", col_nda), ("question", col_question), ("value", col_value),
                      ("date_adm", col_date_adm), ("sex", col_sex), ("age", col_age), ("date_triage_end", col_date_triage_end)]:
        print(f"  {name:12} → {col}")

    # --- Map question labels ---
    df_raw["question_std"] = df_raw[col_question].apply(match_question)

    print("\n=== Question label mapping check ===")
    print(df_raw.groupby(col_question)["question_std"].first().to_string())

    # --- Verify 'Date et heure' is captured ---
    print("\n=== Verification: 'Date et heure' pattern capture ===")
    mask = df_raw[col_question].str.contains(r"Date\s*et\s*heure", case=False, na=False)
    print(df_raw[mask][col_question].value_counts().to_string())

    df_known = df_raw[df_raw["question_std"].notna()].copy()
    print(f"\nRows after question filtering: {len(df_known):,} / {len(df_raw):,}")

    # --- Pivot long → wide ---
    patient_cols = [c for c in [col_nda, col_date_adm, col_sex, col_age, col_year, col_uam, col_date_triage_end, "source_file"] if c]

    df_pivot = df_known.pivot_table(
        index=patient_cols,
        columns="question_std",
        values=col_value,
        aggfunc="first"
    ).reset_index()
    df_pivot.columns.name = None

    # --- Rename patient columns ---
    rename_map = {}
    if col_nda:      rename_map[col_nda]      = "nda"
    if col_date_adm: rename_map[col_date_adm] = "date_adm_ioa_file"
    if col_sex:      rename_map[col_sex]       = "sex_ioa"
    if col_age:      rename_map[col_age]       = "age_ioa"
    if col_date_triage_end: rename_map[col_date_triage_end] = "date_hour_triage_end"
    df_pivot.rename(columns=rename_map, inplace=True)

    # --- Cleaning ---
    df_pivot["date_adm_ioa_file"] = pd.to_datetime(df_pivot["date_adm_ioa_file"], errors="coerce")
    df_pivot["age_ioa"]      = df_pivot["age_ioa"].astype(str).str.extract(r'(\d+)').astype(float)
    df_pivot["nda"]          = df_pivot["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

    if "date_hour_triage_begin" in df_pivot.columns:
        df_pivot["date_hour_triage_begin"] = pd.to_datetime(df_pivot["date_hour_triage_begin"], errors="coerce")

    # --- Triage duration ---
    if "date_hour_triage_begin" in df_pivot.columns and "date_hour_triage_end" in df_pivot.columns:
        df_pivot["date_hour_triage_end"] = pd.to_datetime(df_pivot["date_hour_triage_end"], errors="coerce")
        df_pivot["duration_triage_ioa_min"] = (
            df_pivot["date_hour_triage_end"] - df_pivot["date_hour_triage_begin"]
        ).dt.total_seconds() / 60
        print(f"\n=== Triage duration (min) ===")
        print(df_pivot["duration_triage_ioa_min"].describe().round(1))

    # --- Filter 2021–2025 ---
    df_pivot = df_pivot[
        (df_pivot["date_adm_ioa_file"].dt.year >= YEAR_MIN) &
        (df_pivot["date_adm_ioa_file"].dt.year <= YEAR_MAX)
    ].copy()
    print(f"\nRows after year filter ({YEAR_MIN}-{YEAR_MAX}): {len(df_pivot):,}")

    # --- Hospital ---
    df_pivot["hospital"] = np.where(
        df_pivot["source_file"].str.contains("SA", case=False, na=False), "SA", "PEL"
    )

    # --- Year column ---
    df_pivot["year"] = df_pivot["date_adm_ioa_file"].dt.year

    # --- Triage harmonization ---
    if "triage_raw" in df_pivot.columns:
        df_pivot["triage"] = df_pivot["triage_raw"].apply(harmonize_triage)

    # --- Chief complaint cleaning ---
    if "chief_complaint" in df_pivot.columns:
        df_pivot["chief_complaint"] = df_pivot["chief_complaint"].astype(str).str.replace(r".*-\s*", "", regex=True)
        df_pivot["chief_complaint"] = df_pivot["chief_complaint"].replace(['nan', 'None', ''], 'Unknown')

    # --- Final columns ---
    final_cols = [c for c in [
        "nda", "age_ioa", "sex_ioa", "hospital", "year", "date_adm_ioa_file",
        "date_hour_triage_begin", "date_hour_triage_end", "duration_triage_ioa_min",
        "triage", "triage_raw", "transport", "chief_complaint",
        "anam_ioa", "atcd_ioa", "ttt_adm_ioa_file",
        "admission_summary_ioa", "evolution_ioa"
    ] if c in df_pivot.columns]

    df_final = df_pivot[final_cols].sort_values("date_adm_ioa_file").drop_duplicates(subset="nda")

    # --- Export ---

    df_final.to_csv(output_file, index=False)

    print(f"\n=== Triage score distribution ===")
    print(df_final["triage"].value_counts(dropna=False).sort_index())

    print(f"\n=== Patients per hospital and year ===")
    print(df_final.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

    print(f"\n🚀 Done! {len(df_final):,} unique patients ({YEAR_MIN}-{YEAR_MAX}).")
    print(f"📁 Saved to: {output_file}")
    print(f"Columns: {df_final.columns.tolist()}")

    return df_final

# ===========================
if __name__ == "__main__":
    df_final = main()

# ============================================
# INVESTIGATION 1: True NaN (triage AND triage_raw = NaN)
# ============================================
true_nan = df_final[df_final["triage"].isna() & df_final["triage_raw"].isna()]

print(f"\n{'='*55}")
print(f"  INVESTIGATION 1: True NaN (triage AND triage_raw = NaN)")
print(f"{'='*55}")
print(f"Total patients             : {len(df_final):,}")
print(f"True NaN (no triage at all): {len(true_nan):,} ({len(true_nan)/len(df_final)*100:.1f}%)")

print("\n--- True NaN per hospital ---")
print(true_nan["hospital"].value_counts(dropna=False).to_string())

print("\n--- True NaN per year ---")
print(true_nan["year"].value_counts(dropna=False).sort_index().to_string())

print("\n--- True NaN per hospital AND year ---")
print(true_nan.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

# ============================================
# INVESTIGATION 2: NaN in date_hour_triage_begin
# ============================================
print(f"\n{'='*55}")
print(f"  INVESTIGATION 2: NaN in date_hour_triage_begin")
print(f"{'='*55}")

if "date_hour_triage_begin" in df_final.columns:
    nan_triage_begin = df_final[df_final["date_hour_triage_begin"].isna()]

    print(f"Total patients                   : {len(df_final):,}")
    print(f"NaN date_hour_triage_begin        : {len(nan_triage_begin):,} ({len(nan_triage_begin)/len(df_final)*100:.1f}%)")

    print("\n--- NaN date_hour_triage_begin per hospital ---")
    print(nan_triage_begin["hospital"].value_counts(dropna=False).to_string())

    print("\n--- NaN date_hour_triage_begin per year ---")
    print(nan_triage_begin["year"].value_counts(dropna=False).sort_index().to_string())

    print("\n--- NaN date_hour_triage_begin per hospital AND year ---")
    print(nan_triage_begin.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

    print("\n--- Fill rate (%) of date_hour_triage_begin per hospital AND year ---")
    total  = df_final.groupby(["hospital", "year"]).size().unstack(fill_value=0)
    manque = nan_triage_begin.groupby(["hospital", "year"]).size().unstack(fill_value=0)
    taux   = ((1 - manque / total) * 100).round(1)
    print(taux.to_string())
else:
    print("⚠️  Column 'date_hour_triage_begin' not found — check the exact label in source files.")

--- INVENTORY: 4 files detected ---
 ✅ Loaded: CGJ 030a - Dossier IOA complet - 2022.xlsx
 ✅ Loaded: CGJ 030a - Dossier IOA complet - 2023.xlsx
 ✅ Loaded: CGJ 030a - Dossier IOA complet - 2024.xlsx
 ✅ Loaded: CGJ 030a - Dossier IOA complet - 2025.xlsx
Total rows loaded: 3046809
col_date_triage_end → Date création questionnaire
patient_cols sera → ['NDA', 'Date Entrée Séjour avec heure', 'Sexe patient', "Age à l'entrée du séjour", 'Code UAM entrée séjour', 'Date création questionnaire', 'source_file']

=== Column mapping detected ===
  nda          → NDA
  question     → Libellé question
  value        → Libellé court ou long
  date_adm     → Date Entrée Séjour avec heure
  sex          → Sexe patient
  age          → Age à l'entrée du séjour
  date_triage_end → Date création questionnaire

=== Question label mapping check ===
Libellé question
A jeun                                                                         None
Aide(s) à domicile                                           

In [59]:
# Voir les triage_raw non mappés
print("=== triage_raw values NOT mapped (NaN after harmonize) ===")
mask_nan = df_final["triage"].isna() & df_final["triage_raw"].notna()
print(df_final[mask_nan]["triage_raw"].value_counts().to_string())

print("\n=== ALL triage_raw values ===")
print(df_final["triage_raw"].value_counts(dropna=False).to_string())

=== triage_raw values NOT mapped (NaN after harmonize) ===
Series([], )

=== ALL triage_raw values ===
triage_raw
Urgent (médecin <1h)                   38860
Peu urgent (médecin <2h)               35956
Très urgent (médecin <20min)           25624
Médecin <90min, ouis IDE si besoin     17868
Médecin <120min, puis IDE si besoin    16024
Non urgent (médecin <3h)                9424
Infirmière <10min Médecin < 20min       9219
Médecin <60min, puis IDE si besoin      8319
Médecin <240min                         6452
Sans délai (IDE et Médecin)              699
Réanimation (médecin <1min)              497
NaN                                       26


In [60]:
# ============================================
# INVESTIGATION 3: triage_raw values that failed mapping (triage = NaN but triage_raw is not NaN)
# ============================================
print(f"\n{'='*55}")
print(f"  INVESTIGATION 3: triage_raw values not mapped")
print(f"{'='*55}")

if "triage_raw" in df_final.columns:
    unmapped = df_final[df_final["triage"].isna() & df_final["triage_raw"].notna()]
    print(f"Patients with triage_raw filled but triage = NaN: {len(unmapped):,}")

    print("\n--- Unmapped triage_raw values (sorted by frequency) ---")
    print(unmapped["triage_raw"].value_counts(dropna=False).to_string())


  INVESTIGATION 3: triage_raw values not mapped
Patients with triage_raw filled but triage = NaN: 0

--- Unmapped triage_raw values (sorted by frequency) ---
Series([], )


In [61]:
print(df_final['triage'].value_counts(dropna=False).to_string())

print("=== Triage distribution par année ===")
print(df_final.groupby(["year", "triage"])["nda"].count().unstack(fill_value=0).to_string())

print("\n=== Triage distribution par année (%) ===")
triage_pct = (
    df_final.groupby(["year", "triage"])["nda"]
    .count()
    .unstack(fill_value=0)
    .apply(lambda row: (row / row.sum() * 100).round(1), axis=1)
)
print(triage_pct.to_string())

triage
3      65047
4      51980
2      34843
5      15876
1       1196
NaN       26
=== Triage distribution par année ===
triage    1     2      3      4     5
year                                 
2022    176  9101  15710  14844  3592
2023    145  9603  13494  11817  2901
2024    362  9209  15097  12760  4337
2025    513  6930  20746  12559  5046

=== Triage distribution par année (%) ===
triage    1     2     3     4     5
year                               
2022    0.4  21.0  36.2  34.2   8.3
2023    0.4  25.3  35.5  31.1   7.6
2024    0.9  22.0  36.1  30.6  10.4
2025    1.1  15.1  45.3  27.4  11.0


In [50]:
print("\n--- All triage_raw unique values ---")
print(df_final["triage_raw"].value_counts(dropna=False).to_string())


--- All triage_raw unique values ---
triage_raw
Urgent (médecin <1h)                   38860
Peu urgent (médecin <2h)               35956
Très urgent (médecin <20min)           25624
Médecin <90min, ouis IDE si besoin     17868
Médecin <120min, puis IDE si besoin    16024
Non urgent (médecin <3h)                9424
Infirmière <10min Médecin < 20min       9219
Médecin <60min, puis IDE si besoin      8319
Médecin <240min                         6452
Sans délai (IDE et Médecin)              699
Réanimation (médecin <1min)              497
NaN                                       26


In [ ]:
# #===================================================
# # IOA TRIAGE FILES - HARMONIZED PROCESSING (2021-2025)
# # New format: long → wide (no skiprows, named columns)
# #===================================================
# import os
# import glob
# import re
# import pandas as pd
# import numpy as np
# import unidecode
#
# # ----------------------------
# # CONSTANTS
# # ----------------------------
# DATA_PATH = "../Datanad/subset_data"
# YEAR_MIN = 2021
# YEAR_MAX = 2025
# TRIAGE_THRESHOLD_MAX = 1440
#
# # ----------------------------
# # 1️⃣ FILE INVENTORY
# # ----------------------------
# def get_ioa_files(pattern="CGJ 030*"):
#     files = glob.glob(os.path.join(DATA_PATH, pattern))
#     files = [f for f in files if not os.path.basename(f).startswith("~$")]
#     print(f"--- INVENTORY: {len(files)} files detected ---")
#     return sorted(files)
#
# # ----------------------------
# # 2️⃣ LOAD AND CONCATENATE
# # ----------------------------
# def load_files(files):
#     dfs = []
#     for f in files:
#         try:
#             df_temp = pd.read_excel(f, header=0)
#             df_temp.columns = df_temp.columns.astype(str).str.strip()
#             df_temp['source_file'] = os.path.basename(f)
#             dfs.append(df_temp)
#             print(f" ✅ Loaded: {os.path.basename(f)}")
#         except Exception as e:
#             print(f" ❌ Error: {os.path.basename(f)} : {str(e)[:50]}")
#     df = pd.concat(dfs, ignore_index=True)
#     print(f"Total rows loaded: {len(df)}")
#     return df
#
# # ----------------------------
# # 3️⃣ MAP QUESTION LABELS → STANDARD COLUMN NAMES
# # ----------------------------
# questions_mapping = {
#     "triage_raw":             [r"Tri\s*IAO", r"Score\s*de\s*Gravit[ée]", r"Niveau\s*de\s*Triage"],
#     "transport":              [r"Mode\s*d.arriv[ée]e", r"Mode\s*de\s*Transport"],
#     "atcd_ioa":               [r"Ant[ée]c[ée]dents"],
#     "chief_complaint":        [r"Motifs?\s*de\s*recours", r"Motif\s*de\s*consultation"],
#     "anam_ioa":               [r"Circonstances", r"Commentaires\s*aux\s*urgences",
#                                r"Histoire\s*de\s*la\s*maladie", r"Anamn[èe]se"],
#     "ttt_adm_ioa":            [r"Traitement\s*administr[ée]", r"Traitement\s*en\s*cours"],
#     "admission_summary_ioa":  [r"Synth[èe]se\s*PEC", r"Synth[èe]se\s*Initiale"],
#     "evolution_ioa":          [r"Evolution"],
# }
#
# def match_question(label):
#     if pd.isna(label):
#         return None
#     s = str(label).strip()
#     for std_name, patterns in questions_mapping.items():
#         for pattern in patterns:
#             if re.search(pattern, s, re.IGNORECASE):
#                 return std_name
#     return None
#
# # ----------------------------
# # 4️⃣ IDENTIFY COLUMN NAMES
# # ----------------------------
# def find_col(df, patterns):
#     for p in patterns:
#         for col in df.columns:
#             if re.search(p, str(col), re.IGNORECASE):
#                 return col
#     return None
#
# # ----------------------------
# # 5️⃣ HARMONIZE TRIAGE SCORE
# # ----------------------------
# def harmonize_triage(x):
#     if pd.isna(x): return np.nan
#     s = unidecode.unidecode(str(x)).lower().strip()
#     if any(k in s for k in ["reanim", "sans delai"]):        return "1"
#     if "tres urgent" in s or "medecin < 20min" in s:         return "2"
#     if "peu urgent" in s or "medecin < 2h" in s:             return "4"
#     if "non urgent" in s or "medecin < 3h" in s:             return "5"
#     if any(k in s for k in ["urgent", "1h", "60min"]):       return "3"
#     return np.nan
#
# # ----------------------------
# # 6️⃣ MAIN SCRIPT
# # ----------------------------
# def main():
#     files = get_ioa_files()
#     df_raw = load_files(files)
#
#     # --- Identify key columns ---
#     col_nda        = find_col(df_raw, [r"NDA", r"N°\s*DA", r"N°\s*dossier"])
#     col_question   = find_col(df_raw, [r"Libell[ée]\s*question"])
#     col_value      = find_col(df_raw, [r"Libell[ée]\s*court\s*ou\s*long", r"Libell[ée]\s*long"])
#     col_date_adm   = find_col(df_raw, [r"Date\s*Entr[ée]e\s*S[ée]jour\s*avec\s*heure", r"Date\s*Entr[ée]e\s*S[ée]jour"])
#     col_sex        = find_col(df_raw, [r"Sexe"])
#     col_age        = find_col(df_raw, [r"Age"])
#     col_year       = find_col(df_raw, [r"Ann[ée]e\s*entr[ée]e"])
#     col_uam        = find_col(df_raw, [r"Code\s*UAM"])
#
#     print("\n=== Column mapping detected ===")
#     for name, col in [("nda", col_nda), ("question", col_question), ("value", col_value),
#                       ("date_adm", col_date_adm), ("sex", col_sex), ("age", col_age)]:
#         print(f"  {name:12} → {col}")
#
#     # --- Map question labels ---
#     df_raw["question_std"] = df_raw[col_question].apply(match_question)
#
#     print("\n=== Question label mapping check ===")
#     print(df_raw.groupby(col_question)["question_std"].first().to_string())
#
#     df_known = df_raw[df_raw["question_std"].notna()].copy()
#     print(f"\nRows after filtering: {len(df_known)} / {len(df_raw)}")
#
#     # --- Pivot long → wide ---
#     patient_cols = [c for c in [col_nda, col_date_adm, col_sex, col_age, col_year, col_uam, "source_file"] if c]
#
#     df_pivot = df_known.pivot_table(
#         index=patient_cols,
#         columns="question_std",
#         values=col_value,
#         aggfunc="first"
#     ).reset_index()
#     df_pivot.columns.name = None
#
#     # --- Rename patient columns ---
#     rename_map = {}
#     if col_nda:       rename_map[col_nda]      = "nda"
#     if col_date_adm:  rename_map[col_date_adm] = "date_adm_ioa"
#     if col_sex:       rename_map[col_sex]       = "sex_ioa"
#     if col_age:       rename_map[col_age]       = "age_ioa"
#     df_pivot.rename(columns=rename_map, inplace=True)
#
#     # --- Cleaning ---
#     df_pivot["date_adm_ioa"] = pd.to_datetime(df_pivot["date_adm_ioa"], errors="coerce")
#     df_pivot["age_ioa"] = df_pivot["age_ioa"].astype(str).str.extract(r'(\d+)').astype(float)
#     df_pivot["nda"] = df_pivot["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#
#     # --- Filter 2021–2025 ---
#     df_pivot = df_pivot[
#         (df_pivot["date_adm_ioa"].dt.year >= YEAR_MIN) &
#         (df_pivot["date_adm_ioa"].dt.year <= YEAR_MAX)
#     ].copy()
#     print(f"\nRows after year filter ({YEAR_MIN}-{YEAR_MAX}): {len(df_pivot):,}")
#
#     # --- Hospital ---
#     df_pivot["hospital"] = np.where(df_pivot["source_file"].str.contains("SA", case=False, na=False), "SA", "PEL")
#
#     # --- Triage harmonization ---
#     if "triage_raw" in df_pivot.columns:
#         df_pivot["triage"] = df_pivot["triage_raw"].apply(harmonize_triage)
#
#     # --- Chief complaint cleaning ---
#     if "chief_complaint" in df_pivot.columns:
#         df_pivot["chief_complaint"] = df_pivot["chief_complaint"].astype(str).str.replace(r".*-\s*", "", regex=True)
#         df_pivot["chief_complaint"] = df_pivot["chief_complaint"].replace(['nan', 'None', ''], 'Unknown')
#
#     # --- Final columns ---
#     final_cols = [c for c in [
#         "nda", "age_ioa", "sex_ioa", "hospital", "date_adm_ioa",
#         "triage", "triage_raw", "transport", "chief_complaint",
#         "anam_ioa", "atcd_ioa", "ttt_adm_ioa",
#         "admission_summary_ioa", "evolution_ioa"
#     ] if c in df_pivot.columns]
#
#     df_final = df_pivot[final_cols].sort_values("date_adm_ioa").drop_duplicates(subset="nda")
#
#     # --- Export ---
#     os.makedirs("Data", exist_ok=True)
#     output_file = f"Data/df_ioa_{YEAR_MIN}_{YEAR_MAX}.csv"
#     df_final.to_csv(output_file, index=False)
#
#     print(f"\n=== Triage distribution ===")
#     print(df_final["triage"].value_counts(dropna=False).sort_index())
#     print(f"\n=== Patients per hospital et per year ===")
#     df_final["year"] = df_final["date_adm_ioa"].dt.year
#     print(df_final.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())
#
#     print(f"\n🚀 Done! {len(df_final):,} unique patients ({YEAR_MIN}-{YEAR_MAX}).")
#     print(f"📁 Saved to: {output_file}")
#     print(f"Columns: {df_final.columns.tolist()}")
#
#     return df_final
#
# # ===========================
# if __name__ == "__main__":
#     df_final = main()
#
# # ============================================
# # Nan investigation (triage AND triage_raw = NaN)
# # ============================================
# true_nan = df_final[df_final["triage"].isna() & df_final["triage_raw"].isna()]
#
# print(f"\nPatients with triage AND triage_raw = NaN : {len(true_nan):,}")
#
# print("\n--- NaN per hospital ---")
# print(true_nan["hospital"].value_counts(dropna=False).to_string())
#
# print("\n--- NaN per année ---")
# print(true_nan["year"].value_counts(dropna=False).sort_index().to_string())
#
# print("\n--- NaN per hospital AND year ---")
# print(true_nan.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

### 2022 only

In [ ]:
# #===================================================
# # IOA TRIAGE FILES - HARMONIZED PROCESSING (2022)
# #===================================================
# import os
# import glob
# import re
# import pandas as pd
# import numpy as np
# import unidecode
#
# # ----------------------------
# # CONSTANTS
# # ----------------------------
# DATA_PATH = "../share/new_data_urgences"
# TRIAGE_THRESHOLD_MAX = 1440  # max duration in minutes (24h)
# YEAR_FILTER = 2022
#
# # ----------------------------
# # 1️⃣ FILE INVENTORY
# # ----------------------------
# def get_ioa_files(pattern="CGJ 030*"):
#     """Return sorted list of Excel files excluding temp files."""
#     files = glob.glob(os.path.join(DATA_PATH, pattern))
#     files = [f for f in files if not os.path.basename(f).startswith("~$")]
#     print(f"--- INVENTORY: {len(files)} files detected ---")
#     return sorted(files)
#
# # ----------------------------
# # 2️⃣ LOAD AND CONCATENATE
# # ----------------------------
# def load_files(files, skiprows=3):
#     dfs = []
#     for f in files:
#         try:
#             df_temp = pd.read_excel(f, skiprows=skiprows)
#             df_temp['source_file'] = os.path.basename(f)
#             dfs.append(df_temp)
#             print(f" ✅ Loaded: {os.path.basename(f)}")
#         except Exception as e:
#             print(f" ❌ Error loading {os.path.basename(f)} : {str(e)[:50]}")
#     return pd.concat(dfs, ignore_index=True)
#
# # ----------------------------
# # 3️⃣ SMART MERGE (COLUMN MAPPING)
# # ----------------------------
# def smart_merge(df, mapping):
#     new_df = pd.DataFrame()
#     for target_col, patterns in mapping.items():
#         series = None
#         for p in patterns:
#             matching_cols = [c for c in df.columns if re.search(p, str(c), re.IGNORECASE)]
#             for col in matching_cols:
#                 series = df[col] if series is None else series.combine_first(df[col])
#         new_df[target_col] = series
#     return new_df
#
# # ----------------------------
# # 4️⃣ HARMONIZE TRIAGE SCORE
# # ----------------------------
# def harmonize_triage(x):
#     if pd.isna(x): return np.nan
#
#     # removing accents, converting to lowercase and stripping spaces for easier matching
#     s = unidecode.unidecode(str(x)).lower().strip()
#
#     # 1. looking for specific keywords that indicate the level of urgency, starting with the most specific ones to avoid confusion (ex: "tres urgent" contains "urgent" but is level 2, not 3)
#     if any(k in s for k in ["reanim", "sans delai"]):
#         return "1"
#     if "tres urgent" in s or "medecin < 20min" in s:
#         return "2"
#     if "peu urgent" in s or "medecin < 2h" in s or "120min" in s:
#         return "4"
#     if "non urgent" in s or "medecin < 3h" in s or "240min" in s:
#         return "5"
#     # 2. only now looking for "urgent" because it can be present in "tres urgent" (level 2) or "peu urgent" (level 4), so if we find it here, it's necessarily level 3.
#     if any(k in s for k in ["urgent", "1h", "60min", "90min"]):
#         return "3"
#     return np.nan
#
# # ----------------------------
# # 5️⃣ CLEANING AND PROCESSING DATES / DURATION
# # ----------------------------
# def clean_dates(df, date_cols):
#     for col in date_cols:
#         df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
#     df['duration_triage_ioa_min'] = (df["date_tri_ioa_end"] - df["date_tri_ioa_begin"]).dt.total_seconds() / 60
#     mask = (df['duration_triage_ioa_min'] < 0) | (df['duration_triage_ioa_min'] > TRIAGE_THRESHOLD_MAX)
#     df.loc[mask, 'duration_triage_ioa_min'] = np.nan
#     df = df[df["date_adm_ioa_file"].dt.year == YEAR_FILTER].copy()
#     df["nda"] = df["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#     df["age_ioa"] = df["age_ioa"].astype(str).str.extract(r'(\d+)').astype(float)
#     return df
#
# # ----------------------------
# # 6️⃣ POST PROCESSING / FINAL COLUMNS
# # ----------------------------
# def finalize(df):
#     # Hospital column
#     df['hospital'] = np.where(df['source_file'].str.contains('SA', case=False, na=False), 'SA', 'PEL')
#
#     # Harmonize triage
#     df['triage'] = df['triage_raw'].apply(harmonize_triage)
#
#     # Merge anamnesis columns
#     df['anam_ioa'] = df['anam_2'].combine_first(df['anam_1']).combine_first(df['anam_3'])
#
#     # Clean chief complaint
#     df['chief_complaint'] = df['chief_complaint'].astype(str).str.replace(r".*-\s*", "", regex=True)
#     df['chief_complaint'] = df['chief_complaint'].replace(['nan', 'None', ''], 'Unknown')
#
#     # Select final columns
#     final_cols = [
#         "nda","age_ioa","sex_ioa","hospital","date_adm_ioa_file",
#         "date_tri_ioa_begin","date_tri_ioa_end","duration_triage_ioa_min",
#         "chief_complaint","triage","triage_raw","transport","anam_ioa","atcd_ioa", "admission_summary_ioa", "evolution_ioa"
#     ]
#     df_final = df[final_cols].sort_values(by="date_adm_ioa_file").drop_duplicates(subset="nda")
#     return df_final
#
# # ----------------------------
# # 7️⃣ MAIN SCRIPT
# # ----------------------------
# def main():
#     files = get_ioa_files()
#     df_raw = load_files(files)
#
#     columns_mapping = {
#         "date_adm_ioa_file": [r"Unnamed:\s*2", r"Date\s*d'admission"],
#         "nda": [r"Unnamed:\s*3", r"N°\s*DA", r"NDA"],
#         "sex_ioa": [r"Unnamed:\s*6", r"Sexe"],
#         "age_ioa": [r"Unnamed:\s*7", r"Age"],
#         "date_tri_ioa_end": [r"Unnamed:\s*9", r"Date\s*du\s*Tri"],
#         "date_tri_ioa_begin": [r"Date\s*et\s*heure"],
#         "triage_raw": [r"Tri\s*IAO", r"Score\s*de\s*Gravit[ée]"],
#         "transport": [r"Mode\s*d'arriv[ée]e", r"Mode\s*de\s*Transport"],
#         "atcd_ioa": [r"Ant[ée]c[ée]dents"],
#         "ttt_adm_ioa": [r"Traitement\s*administr[ée]"],
#         "chief_complaint": [r"Motifs?\s*de\s*recours", r"Motif"],
#         "anam_1": [r"Circonstances"],
#         "anam_2": [r"Commentaires\s*aux\s*urgences"],
#         "anam_3": [r"Histoire\s*de\s*la\s*maladie"],
#         "admission_summary_ioa": [r"Synthèse\s*PEC\s*Initiale"],
#         "evolution_ioa": [r"Evolution"]
#     }
#
#     df_mapped = smart_merge(df_raw, columns_mapping)
#     df_mapped['source_file'] = df_raw['source_file']
#
#     df_clean = clean_dates(df_mapped, ["date_adm_ioa_file","date_tri_ioa_begin","date_tri_ioa_end"])
#     df_final = finalize(df_clean)
#
#     # Export
#     df_final.to_csv("df_ioa_subset22pel.csv", index=False)
#
#     # Summary
#     print(f"\n--- Harmonized Triage Score Distribution (2022) ---")
#     print(df_final["triage"].value_counts(dropna=False).sort_index())
#     print(f"\n🚀 Done! {len(df_final)} unique patients processed.")
#
#     return df_final
#
# # ===========================
# if __name__ == "__main__":
#     df_final = main()

In [ ]:
# # Detect suspicious durations
# df_suspicious = df_final[
#     (df_final['duration_triage_ioa_min'] > 180) |
#     (df_final['duration_triage_ioa_min'] < 0)
# ]
#
# print(f"Suspicious durations: {len(df_suspicious)} rows")
# display(df_suspicious.head())

In [ ]:
# # _________________________________________
# # Analysis of the column structure in each file
# #_____________________________________________
#
#
# print("--- files structure analysis ---")
#
# # for file in files:
#     try:
#         # only read from line 4
#         df_cols = pd.read_excel(file, skiprows=3, nrows=0)
#         cols = df_cols.columns.tolist()
#
#         print(f"\n📄 FILE : {os.path.basename(file)}")
#         print(f"Total number of column : {len(cols)}")
#         print("Column list (Index : Nom) :")
#
#         for i, col in enumerate(cols):
#             print(f"  {i} : {col}")
#
#     except Exception as e:
#         print(f"❌ Error on {os.path.basename(file)} : {e}")

In [ ]:
# # 1. filtering where triage score is nan
# echecs_tri = df_final[df_final['triage'].isna()]
#
# # 2. looking at the raw triage score values to understand why they were not recognized by the harmonization function
# print("--- non recognized values analysis 'triage_raw' ---")
# print(echecs_tri['triage_raw'].value_counts(dropna=False).head(20))
#
# # 3. Sample to read context (sans source_file qui a été supprimé)
# print("\n--- failed observation sample ---")
# colonnes_dispo = [c for c in ['tri_raw', 'chief_complaint', 'hospital'] if c in echecs_tri.columns]
# print(echecs_tri[colonnes_dispo].head(20))

In [ ]:
# # ============================================
# # INSPECTION DES VRAIS NaN (triage_raw aussi = NaN)
# # ============================================
#
# true_nan = df_final[df_final["triage"].isna() & df_final["triage_raw"].isna()]
#
# print(f"Patients avec triage ET triage_raw = NaN : {len(true_nan):,}")
#
# print("\n--- NaN par hôpital ---")
# print(true_nan["hospital"].value_counts(dropna=False).to_string())
#
# print("\n--- NaN par année ---")
# true_nan["year"] = pd.to_datetime(true_nan["date_adm_ioa"], errors="coerce").dt.year
# print(true_nan["year"].value_counts(dropna=False).sort_index().to_string())
#
# print("\n--- NaN par hôpital ET année ---")
# print(true_nan.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())


# 3. Medical files #


## New medical files, from 2022 to 2025, long format excel ##

In [47]:
import os
import glob
import pandas as pd
import numpy as np
import re



#------CONSTANTS--------
DATA_PATH = "../Datanad/subset_data"
YEAR_MIN = 2022
YEAR_MAX = 2025
os.makedirs("Datasets", exist_ok=True)
output_file = f"Datasets/df_med_{YEAR_MIN}_{YEAR_MAX}.csv"


# --- Step 1 : Load and read ---
pattern = os.path.join(DATA_PATH, "CGJ_055*")
all_files = glob.glob(pattern)
files = [f for f in all_files if not os.path.basename(f).startswith("~$")]

if not files:
    print("No file found.")
    exit()

dfs = []
for file in files:
    try:
        df = pd.read_excel(file)
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            cols[cols == dup] = [f"{dup}_{i}" if i != 0 else dup for i in range(len(cols[cols == dup]))]
        df.columns = cols
        df['source_file'] = os.path.basename(file)
        dfs.append(df)
        print(f"✅ Loaded : {os.path.basename(file)} ({len(df)} lines)")
    except Exception as e:
        print(f"❌ Error on {os.path.basename(file)} : {type(e).__name__} — {e}")

print(f"\ndfs contient {len(dfs)} dataframe(s)")
print(f"Fichiers tentés : {[os.path.basename(f) for f in files]}")

df_concat = pd.concat(dfs, ignore_index=True)
print(f"\nConcatenation done. Total : {len(df_concat)} lines")

# --- Step 2 : Normalize column names (strip spaces/accents issues) ---
df_concat.columns = df_concat.columns.str.strip()

# Identify the key columns by flexible matching
def find_col(df, pattern):
    """Return first column name matching a regex pattern (case-insensitive)."""
    for col in df.columns:
        if re.search(pattern, str(col), re.IGNORECASE):
            return col
    return None

col_nda        = find_col(df_concat, r"nda")
col_sex        = find_col(df_concat, r"sexe")
col_age = find_col(df_concat, r"age.*entr.e")
col_date_adm = find_col(df_concat, r"date\s+entr.e\s+s.jour\s+avec") # with time
col_date_creation = find_col(df_concat, r"date\s+cr.ation")
col_diag       = find_col(df_concat, r"diagnostic")
col_question   = find_col(df_concat, r"libell.\s+question")                # question label  → future column names
col_value      = find_col(df_concat, r"libell.\s+(court|long)")            # answer content  → future values
col_uam        = find_col(df_concat, r"uam")


print("\n--- Detected columns ---")
for name, val in [("NDA", col_nda), ("Sex", col_sex), ("Admission date", col_date_adm), ("Medical visit date", col_date_creation),
                  ("Diagnostic", col_diag), ("Question label", col_question),
                  ("Answer value", col_value)]:
    print(f"  {name:15s} → {val}")

# --- Step 3 : Pivot long → wide ---
# The pivot key is NDA (one patient = one row after pivot)
# Each unique value of col_question becomes a column, filled with col_value

# Keep metadata columns that are constant per NDA (one value per patient)
meta_cols = [c for c in [col_nda, col_sex, col_age, col_date_adm, col_date_creation, col_diag, col_uam, "source_file"]
             if c is not None]

# Deduplicate metadata (keep first occurrence per NDA)
df_meta = df_concat[meta_cols].drop_duplicates(subset=[col_nda])

# Pivot the question/answer pairs
if col_question and col_value and col_nda:
    df_pivot = df_concat[[col_nda, col_question, col_value]].copy()
    df_pivot = df_pivot.dropna(subset=[col_question])

    # Normalize question labels for clean column names
    df_pivot[col_question] = (df_pivot[col_question]
                               .astype(str)
                               .str.strip()
                               .str.lower()
                               .str.replace(r'\s+', '_', regex=True)
                               .str.replace(r'[^a-z0-9_àâäéèêëîïôùûü]', '', regex=True))

    # Keep last non-null value if a patient has multiple entries for the same question
    df_pivot = df_pivot.drop_duplicates(subset=[col_nda, col_question], keep='last')

    df_wide = df_pivot.pivot(index=col_nda, columns=col_question, values=col_value).reset_index()
    df_wide.columns.name = None
else:
    print("⚠️  Could not find question/value/NDA columns — pivot skipped.")
    df_wide = pd.DataFrame({col_nda: df_concat[col_nda].unique()})

# --- Step 4 : Merge metadata + pivoted questions ---
df_merged = df_meta.merge(df_wide, on=col_nda, how='left')

# --- Step 5 : Rename to standard column names ---
# Map detected metadata columns
rename_map = {}
if col_nda:        rename_map[col_nda]        = "nda"
if col_sex:        rename_map[col_sex]        = "sex"
if col_age:        rename_map[col_age] = "age"
if col_date_adm: rename_map[col_date_adm] = "date_adm_med"
if col_date_creation:  rename_map[col_date_creation]  = "date_hour_medical_visit"
if col_diag:       rename_map[col_diag]       = "diag"


df_merged.rename(columns=rename_map, inplace=True)

# --- Step 6 : Map pivoted question columns → standard names ---
# These regexes match the normalized question labels created during pivot
question_col_mapping = {
    "anam_ed":        [r"histoire", r"anamn.se"],
    "atcd_med":       [r"ant.c.dent", r"atcd"],
    "rx_home":        [r"traitement.*(habituel|entr.e|domicile)"],
    "clinical_exam":  [r"examen.clinique"],
    "rx_ed":          [r"traitement.*administr", r"actes.*th.rapeutiques", r"soins.*urgence"],
    "evolution":      [r".volution"],
    "conclusion":     [r"conclusion"],
    "ccmu":           [r"ccmu"],
    "disposition_med":    [r"devenir"],
    "additional_tests": [r"examens?.compl.mentaires?"]
}

for standard_name, patterns in question_col_mapping.items():
    if standard_name in df_merged.columns:
        continue  # already present from metadata
    for pat in patterns:
        matched = [c for c in df_merged.columns if re.search(pat, str(c), re.IGNORECASE)]
        if matched:
            # Merge all matches into one column (combine_first for non-null priority)
            series = df_merged[matched[0]]
            for extra in matched[1:]:
                series = series.combine_first(df_merged[extra])
            df_merged[standard_name] = series
            break  # stop at first pattern group that matched

# --- Step 7 : Final column selection ---
# We make sure "age" is in the list
final_cols = ["nda", "sex", "age", "date_adm_med", "date_hour_medical_visit",
              "diag", "anam_ed", "atcd_med", "rx_home", "clinical_exam",
              "rx_ed", "evolution", "additional_tests", "conclusion",
              "ccmu", "disposition_med", "source_file"]

df_final = df_merged[[c for c in final_cols if c in df_merged.columns]].copy()

# Clean Age (ensure it's a number)
if "age" in df_final.columns:
    df_final["age"] = pd.to_numeric(df_final["age"], errors='coerce')

# Clean NDA
df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', np.nan)

# --- Step 8 : Export & Saving ---
df_final.to_csv(output_file, index=False)
print(f"\n✅ Saved global file to: {output_file}")

# # --- Step 9 : Subset 2022 for the root environment ---
# print("📦 Creating 2022 medical subset...")
# date_col = "date_adm_med" if "date_adm_med" in df_final.columns else "date_hour_medical_visit"
#
# if date_col in df_final.columns:
#     temp_dates = pd.to_datetime(df_final[date_col], errors='coerce')
#     df_med_2022 = df_final[temp_dates.dt.year == 2022].copy()
#
#     subset_med_csv = "df_med_subset22pel.csv"
#     df_med_2022.to_csv(subset_med_csv, index=False)
#     print(f"✅ Subset 2022 created with {len(df_med_2022)} rows (including age).")

✅ Loaded : CGJ_055b_-_Dossier_patient_full_2023.xlsx (535881 lines)
✅ Loaded : CGJ_055b_-_Dossier_patient_full_2024.xlsx (628960 lines)
✅ Loaded : CGJ_055b_-_Dossier_patient_full_2022.xlsx (574886 lines)
✅ Loaded : CGJ_055b_-_Dossier_patient_full_2025.xlsx (739289 lines)

dfs contient 4 dataframe(s)
Fichiers tentés : ['CGJ_055b_-_Dossier_patient_full_2023.xlsx', 'CGJ_055b_-_Dossier_patient_full_2024.xlsx', 'CGJ_055b_-_Dossier_patient_full_2022.xlsx', 'CGJ_055b_-_Dossier_patient_full_2025.xlsx']

Concatenation done. Total : 2479016 lines

--- Detected columns ---
  NDA             → NDA
  Sex             → Sexe patient
  Admission date  → Date Entrée Séjour avec heure
  Medical visit date → Date création questionnaire
  Diagnostic      → Code et libellé diagnostic
  Question label  → Libellé question
  Answer value    → Libellé court ou long

✅ Saved global file to: Datasets/df_med_2022_2025.csv


In [ ]:
# --- Unique rows check ---
print(f"\n=== UNIQUE ROWS CHECK ===")
print(f"Total rows          : {len(df_final):,}")
print(f"Unique NDA          : {df_final['nda'].nunique():,}")
print(f"Duplicate NDA       : {len(df_final) - df_final['nda'].nunique():,}")

### Old medical files in wide format already in excel ###

In [ ]:
# import os
# import glob
# import pandas as pd
# import numpy as np
# import re
#
#
#
# # --- Step 1 : load and read ---
# #------CONSTANTS--------
# DATA_PATH = "../Datanad/subset_data"
#
# pattern = os.path.join(DATA_PATH, "CGJ_055*")
# all_files = glob.glob(pattern)
# files = [f for f in all_files if not os.path.basename(f).startswith("~$")]
#
#
# if not files:
#     print("no file found.")
#     exit()
#
# dfs = []
# for file in files:
#     try:
#         # We read from line 2 (skiprows=1) to get the actual column names, which are in line 2 of the file
#         df = pd.read_excel(file, skiprows=1)
#
#         # cleaning names to unique columns name to prevent error during the merge (ex: if there are 2 "Unnamed: 1", we rename them "Unnamed: 1" and "Unnamed: 1_2")
#         cols = pd.Series(df.columns)
#         for dup in cols[cols.duplicated()].unique():
#             cols[cols == dup] = [f"{dup}_{i}" if i != 0 else dup for i in range(len(cols[cols == dup]))]
#         df.columns = cols
#
#         # adding the source file name for traceability
#         df['source_file'] = os.path.basename(file)
#
#         dfs.append(df)
#         print(f" ✅ loaded : {os.path.basename(file)} ({len(df)} lignes)")
#     except Exception as e:
#         print(f" ❌ error on {os.path.basename(file)} : {e}")
#
# # Concaténation
# df_concat = pd.concat(dfs, ignore_index=True)
# print(f"\nConcatenation done. Total : {len(df_concat)} lines")
#
# # --- Étape 2 : Mapping ---
# rename_map = {
#     "Unnamed: 1": "nda",
#     "Unnamed: 2": "nom",
#     "Unnamed: 3": "prenom",
#     "Unnamed: 4": "sex",
#     "Unnamed: 5": "age",
#     "Unnamed: 6": "date_adm",
#     "Unnamed: 7": "diag"
# }
# df_concat.rename(columns=rename_map, inplace=True)
#
# # --- Step 3 : Fusion Regex  ---
# def merge_columns_regex(df, regex_candidates, new_col_name):
#     series = df[new_col_name] if new_col_name in df.columns else None
#     for pattern in regex_candidates:
#         # We look dor every column including Unnamed_1, Unnamed_2...
#         matching_cols = [col for col in df.columns if re.search(pattern, str(col), re.IGNORECASE)]
#         for col in matching_cols:
#             if col == new_col_name: continue
#             if series is None:
#                 series = df[col]
#             else:
#                 series = series.combine_first(df[col])
#     if series is not None:
#         df[new_col_name] = series
#     return df
#
# columns_mapping = {
#     "anam_ed": [r"histoire", r"anamn[eéèê]se"],
#     "atcd_med": [r"ant[eé]c[eé]dent", r"atcd"],
#     "rx_home": [r"traitement\s+(habituel|à l'entrée|au domicile)"],
#     "clinical_exam" : [r"Examen clinique initial"],
#     "rx_ed": [r"traitement\s+administr[ée]", r"actes\s+th[ée]rapeutiques", r"soins\s+d'urgence"],
#     "evolution": [r"[eé]volution"],
#     "conclusion": [r"conclusion"],
#     "ccmu": [r"ccmu"],
#     "diag": [r"diagnostic"],
#     "disposition": [r"devenir"],
#     "additional_tests": [r"examens?\s+compl[ée]mentaires?"]
# }
#
# for new_col, regex_list in columns_mapping.items():
#     df_concat = merge_columns_regex(df_concat, regex_list, new_col)
#
# # --- Étape 4 & 5 : column selection ---
# final_cols = ["nda", "sex", "age", "date_adm", "diag",
#               "anam_ed", "atcd_med", "rx_home", "clinical_exam","rx_ed"
#               "evolution", "additional_tests" , "conclusion", "ccmu", "disposition", "source_file"]
#
# df_final = df_concat[[c for c in final_cols if c in df_concat.columns]].copy()
#
# # empty lines removal (lines where nda, diag and anam_urg are all empty) to avoid having too many empty lines in the final file, but we keep lines where only one of these columns is filled because it can still contain useful information (ex: a line with only anam_urg filled can still be useful for text analysis on the anamnese)
# df_final = df_final.dropna(how='all', subset=["nda", "diag", "anam_ed"])
#
# # cleaning NDA
# df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', np.nan)
#
# print("\n--- BILAN PER FILE ---")
# for f in files:
#     name = os.path.basename(f)
#     print(f" - {name} : {len(df_final[df_final['source_file']==name])} valid files.")
#
# df_final.to_csv("df_med_subset22pel.csv", index=False)


# 4. Radio #


## 2022 to 2025 ##

In [ ]:
# ===================================================
# Filtering and pivoting radio exam files 2022-2025
# ===================================================
import pandas as pd
import glob
import os

# --- Paths ---
DATA_PATH = "../Datanad/subset_data"
OUTPUT_PATH = "Datasets"
YEAR_START = 2022
YEAR_END = 2025
output_csv = os.path.join(OUTPUT_PATH, f"df_radio_{YEAR_START}_{YEAR_END}.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Step 1: Load all CGJ 073* files ---
all_files = glob.glob(os.path.join(DATA_PATH, "*"))
files = [f for f in all_files
         if os.path.basename(f).startswith("CGJ 073")
         and not os.path.basename(f).startswith("~$")]

print(f"{len(files)} file(s) found:")
for f in files:
    print(f"  - {os.path.basename(f)}")

# --- Step 2: Load, filter by year range and concatenate ---
dfs = []
for file in files:
    name = os.path.basename(file)
    try:
        df_raw = pd.read_excel(file, header=3)
        df_raw.columns = df_raw.columns.astype(str).str.strip()

        date_col = "Date Entree"
        if date_col not in df_raw.columns:
            print(f"⚠️  '{date_col}' not found in {name} — columns: {df_raw.columns.tolist()}")
            continue

        df_raw[date_col] = pd.to_datetime(df_raw[date_col], dayfirst=True, errors='coerce')
        df_filtered = df_raw[
            (df_raw[date_col].dt.year >= YEAR_START) &
            (df_raw[date_col].dt.year <= YEAR_END)
        ].copy()

        df_filtered["source_file"] = name
        dfs.append(df_filtered)
        print(f"✅ {name} — {len(df_filtered)} rows after {YEAR_START}-{YEAR_END} filter")

    except Exception as e:
        print(f"❌ {name} — {type(e).__name__}: {e}")

if not dfs:
    raise ValueError("No files loaded — check paths and column names.")

df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.astype(str).str.strip()
print(f"\nTotal after concat: {len(df)} rows")

# --- Step 3: Deduplicate exams ---
date_col       = "Date Entree"
exam_label_col = "Libellé examen"
exam_type_col  = "Libellé Type examen"
report_col     = "Compte rendu (cr) pour recherche texte (4000 caractères)"
patient_col    = "Numéro de venue (Xplore)"

# Check all key columns exist
for col in [date_col, exam_label_col, exam_type_col, report_col, patient_col]:
    if col not in df.columns:
        print(f"⚠️  Missing column: '{col}'")

key_cols = [date_col, exam_label_col, exam_type_col]

def keep_report_if_exists(group):
    """For each exam group, keep rows with a report if any exist, otherwise keep first row."""
    if report_col in group.columns:
        has_report = group[report_col].notna() & (group[report_col].astype(str).str.strip() != "")
        if has_report.any():
            return group[has_report]
    return group.iloc[[0]]

df_clean = (df.groupby(key_cols, as_index=False, group_keys=False)
              .apply(keep_report_if_exists)
              .reset_index(drop=True))

# --- Step 4: Rename key columns ---
df_clean.rename(columns={
    patient_col: "nda",
    date_col:    "admission_date"
}, inplace=True)

# --- Step 5: Create indices for each exam type per patient ---
df_clean["exam_index"]     = df_clean.groupby(["nda", exam_type_col]).cumcount() + 1
df_clean["date_col_new"]   = df_clean[exam_type_col] + "_date"   + df_clean["exam_index"].astype(str)
df_clean["exam_col_new"]   = df_clean[exam_type_col] + "_exam"   + df_clean["exam_index"].astype(str)
df_clean["report_col_new"] = df_clean[exam_type_col] + "_report" + df_clean["exam_index"].astype(str)

# --- Step 6: Pivot to wide format ---
df_dates   = df_clean.pivot(index=["nda", "admission_date"], columns="date_col_new",   values="Date heure examen")
df_exams   = df_clean.pivot(index=["nda", "admission_date"], columns="exam_col_new",   values=exam_label_col)
df_reports = df_clean.pivot(index=["nda", "admission_date"], columns="report_col_new", values=report_col)

df_wide = df_dates.join([df_exams, df_reports]).reset_index()
df_wide.columns.name = None

# --- Step 7: Clean NDA ---
df_wide["nda"] = df_wide["nda"].astype(str).str.replace(r'\.0$', '', regex=True)

# --- Step 8: Save ---
df_wide.to_csv(output_csv, index=False)
print(f"\n✅ Done! {len(df_wide)} unique patients saved to {output_csv}")


##

In [ ]:
# ===================================================
# Filtering and pivoting radio exam files for 2022
# ===================================================
import pandas as pd
import os

# --- Paths ---
DATA_PATH = "../Datanad/subset_data"
input_file = "../Datanad/subset_data/CGJ 073 - Radio - 2224.xlsx"
output_excel = "../Datanad/subset_2022_PEL/CGJ 073 - Radio - 22.xlsx"
output_csv = "df_radio_subset22pel.csv"

# --- Step 1: Load Excel with header on row 4 ---
df_radio = pd.read_excel(input_file, header=3)
df_radio.columns = df_radio.columns.astype(str).str.strip()  # clean column names

# --- Step 2: Parse dates and filter 2022 ---
date_col = "Date Entree"
if date_col in df_radio.columns:
    df_radio[date_col] = pd.to_datetime(df_radio[date_col], dayfirst=True, errors='coerce')
    df_filtered = df_radio[df_radio[date_col].dt.year == 2022].copy()
    df_filtered.to_excel(output_excel, index=False)
else:
    raise ValueError(f"Column '{date_col}' not found in the file.")

# --- Step 3: Load filtered file ---
df = pd.read_excel(output_excel)
df.columns = df.columns.astype(str).str.strip()

# --- Step 4: Deduplicate exams ---
date_col = "Date Entree"
exam_label_col = "Libellé examen"
exam_type_col = "Libellé Type examen"
report_col = "Compte rendu (cr) pour recherche texte (4000 caractères)"
patient_col = "Numéro de venue (Xplore)"

key_cols = [date_col, exam_label_col, exam_type_col]

def keep_report_if_exists(group):
    if report_col in group.columns:
        has_report = group[report_col].notna() & (group[report_col].astype(str).str.strip() != "")
        if has_report.any():
            return group[has_report]
    return group.iloc[[0]]

df_clean = df.groupby(key_cols, as_index=False, group_keys=False).apply(keep_report_if_exists).reset_index(drop=True)

# --- Step 5: Rename columns for pivot ---
df_clean.rename(columns={
    patient_col: "nda",
    date_col: "admission_date"
}, inplace=True)

# --- Step 6: Create indices for each exam type per patient ---
df_clean["exam_index"] = df_clean.groupby(["nda", exam_type_col]).cumcount() + 1
df_clean["date_col_new"] = df_clean[exam_type_col] + "_date" + df_clean["exam_index"].astype(str)
df_clean["exam_col_new"] = df_clean[exam_type_col] + "_exam" + df_clean["exam_index"].astype(str)
df_clean["report_col_new"] = df_clean[exam_type_col] + "_report" + df_clean["exam_index"].astype(str)

# --- Step 7: Pivot to wide format ---
df_dates = df_clean.pivot(index=["nda", "admission_date"], columns="date_col_new", values="Date heure examen")
df_exams = df_clean.pivot(index=["nda", "admission_date"], columns="exam_col_new", values=exam_label_col)
df_reports = df_clean.pivot(index=["nda", "admission_date"], columns="report_col_new", values=report_col)

# Merge all pivoted data
df_wide = df_dates.join([df_exams, df_reports]).reset_index()

# --- Step 8: Clean patient ID ---
df_wide["nda"] = df_wide["nda"].astype(str).str.replace(r'\.0$', '', regex=True)

# --- Step 9: Save final CSV ---
df_wide.to_csv(output_csv, index=False)
print(f"✅ Done! {len(df_wide)} unique patients saved in {output_csv}")

In [ ]:
df_wide

# 5. Biology #

### 2022 to 2025 ###

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# --- Config ---
DATA_PATH = "../Datanad/subset_data"
OUTPUT_PATH = "Datasets"
YEAR_START = 2022
YEAR_END = 2025
output_csv = os.path.join(OUTPUT_PATH, f"df_bio_{YEAR_START}_{YEAR_END}.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# #######################################################
# A) LOAD ALL GLIMS FILES
# #######################################################
def append_all_glims():
    all_glims_files = glob.glob(os.path.join(DATA_PATH, "GLI*.xlsx"))
    glims_files = [f for f in all_glims_files if not os.path.basename(f).startswith("~$")]

    if not glims_files:
        print("[GLIMS] No files found.")
        return pd.DataFrame()

    print(f"[GLIMS] {len(glims_files)} file(s) found:")
    for f in glims_files:
        print(f"  - {os.path.basename(f)}")

    dfs = []
    for f in glims_files:
        try:
            df = pd.read_excel(f, sheet_name="Analyses", header=1, dtype={"N° venue": str})

            # Filter by year range
            date_col = "Date Prélèvement (en date)"
            if date_col in df.columns:
                df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
                df = df[
                    (df[date_col].dt.year >= YEAR_START) &
                    (df[date_col].dt.year <= YEAR_END)
                ].copy()

            # Raw value backup with dot decimal
            raw_with_dot = df["Résultat brute de l'analyse"].astype(str).str.replace(',', '.', regex=False)

            # Merge result sources
            df['mixed_result'] = df["Résultat de l'analyse"].astype(str).replace('nan', np.nan)
            df['mixed_result'] = df['mixed_result'].fillna(raw_with_dot)

            df["source_file"] = os.path.basename(f)
            dfs.append(df)
            print(f"✅ {os.path.basename(f)} — {len(df)} rows after {YEAR_START}-{YEAR_END} filter")

        except Exception as e:
            print(f"❌ {os.path.basename(f)} — {type(e).__name__}: {e}")

    if not dfs:
        return pd.DataFrame()

    df_glims_raw = pd.concat(dfs, ignore_index=True)
    print(f"\n[GLIMS raw] Total: {df_glims_raw.shape[0]} rows.")
    return df_glims_raw

# #######################################################
# B) PROCESS GLIMS (PIVOT)
# #######################################################
def process_glims_pivot(df_glims_raw):
    # 1 - Rename columns
    rename_map = {
        "N° venue": "nda",
        "Date Prélèvement (en date)": "sample_date",
        "Libellé analyse détaillée": "analysis",
        "mixed_result": "result",
    }
    df_glims_raw.rename(columns=rename_map, inplace=True, errors="ignore")

    # 2 - Clean IDs and dates
    if "nda" in df_glims_raw.columns:
        df_glims_raw["nda"] = (df_glims_raw["nda"].astype(str)
                                .str.replace(r'\.0$', '', regex=True)
                                .str.strip())
    if "sample_date" in df_glims_raw.columns:
        df_glims_raw["sample_date"] = pd.to_datetime(df_glims_raw["sample_date"], errors="coerce")

    # 3 - Normalize analysis labels
    df_glims_raw["analysis"] = (df_glims_raw["analysis"].astype(str)
                                 .str.replace(r'\xa0', ' ', regex=True)
                                 .str.strip())
    analysis_lower = df_glims_raw["analysis"].str.lower()

    # Force explicit names
    mask_culture = df_glims_raw["analysis"] == "Culture"
    df_glims_raw.loc[mask_culture, "analysis"] = "culture_global"

    mask_gds = (analysis_lower.str.contains("origine", na=False) &
                analysis_lower.str.contains("gds", na=False))
    df_glims_raw.loc[mask_gds, "analysis"] = "gds_origin_global"

    lcr_mapping = {
        "Aspect du LCR": "lcr_aspect_1",
        "Asp.LCR centrifugé": "lcr_aspect_2",
        "Aspect": "lcr_aspect_3"
    }
    df_glims_raw["analysis"] = df_glims_raw["analysis"].replace(lcr_mapping)

    # 4 - Clean result values (vectorized)
    def clean_values_vectorized(df):
        result   = df['result'].astype(str).str.strip()
        analysis = df['analysis'].astype(str)

        mask_culture = analysis == "culture_global"
        mask_lcr     = analysis.isin(["lcr_aspect_1", "lcr_aspect_2", "lcr_aspect_3"])
        mask_gds     = analysis == "gds_origin_global"
        mask_empty   = result.str.lower().isin(["", "nan", "none"])
        mask_nren    = result.str.contains(r'\{<NREN', na=False, regex=True)

        cleaned = result.copy()

        # Culture
        cleaned.loc[mask_culture & mask_empty]            = np.nan
        cleaned.loc[mask_culture & ~mask_empty & ~mask_nren] = "YES"
        cleaned.loc[mask_culture & mask_nren]             = "NREN"

        # Empty (non-culture)
        cleaned.loc[mask_empty & ~mask_culture] = np.nan

        # GDS
        gds_mask_clean = mask_gds & ~mask_nren
        if gds_mask_clean.any():
            cleaned.loc[gds_mask_clean] = (cleaned.loc[gds_mask_clean]
                                            .str.replace('{', '', regex=False)
                                            .str.replace('}', '', regex=False)
                                            .str.replace('<', '', regex=False)
                                            .str.replace('BC_', '', regex=False)
                                            .str.strip())
        cleaned.loc[mask_gds & mask_nren] = "NREN"

        # Numeric extraction
        mask_numeric = ~(mask_culture | mask_lcr | mask_gds | mask_empty | mask_nren)
        if mask_numeric.any():
            numeric_extracted = cleaned.loc[mask_numeric].str.extract(r'(\d+\.?\d*)', expand=False)
            cleaned.loc[mask_numeric] = numeric_extracted.fillna(cleaned.loc[mask_numeric])

        # Numeric NREN
        cleaned.loc[mask_nren & ~(mask_culture | mask_lcr | mask_gds)] = "NREN"

        return cleaned

    df_glims_raw["result"] = clean_values_vectorized(df_glims_raw)

    # 5 - Analysis mapping
    analysis_map = {
        "Créatinine sg": "creatinine", "Urée sg": "urea",
        "Potassium sg": "potassium", "Sodium sg": "sodium", "Calcium sg arsenazo": "calcium",
        "Troponine I HS": "troponine", "CKMB": "ckmb", "BNP": "bnp",
        "ASAT (TGO)": "asat", "ALAT (TGP)": "alat", "Bilirubine totale": "bili_total",
        "PAL sg": "alp", "Lipase sg": "lipase",
        "Leucocytes": "leucocytes", "PNeutro Va": "neutrophils", "Lympho Va": "lymphocytes",
        "Monocytes Va": "monocytes", "PBaso Va": "basophils", "PEosino Va": "eosinophils",
        "Hémoglobine": "hemoglobine", "PlaquettesEDTA": "platelets",
        "TP Taux Prothrombine": "pt", "TCA patient": "aptt", "FibrinogèneClauss": "fibrinogen",
        "Activité AXa (HNF)": "axa_hnf", "Activité AXa Xarelto": "axa_xarelto",
        "Activité AXa Eliquis": "axa_eliquis", "Activité AXa HBPM": "axa_hbpm",
        "Anti-IIa Pradaxa": "aiia_pradaxa", "Activité AXa Arixtra": "axa_arixtra",
        "Activité AXa Orgaran": "axa_orgaran",
        "CK sg": "ck", "D-Dimères dosage": "ddimer",
        "Lactate": "lactates_a", "Lactate plasma vein": "lactates_v",
        "Calcium ionisé": "calcium_ionized",
        "Procalcitonine ser": "pct", "CRP": "crp",
        "Fer sg": "iron", "Ferritine": "ferritin",
        "culture_global": "culture",
        "lcr_aspect_1": "lcr_aspect_1", "lcr_aspect_2": "lcr_aspect_2", "lcr_aspect_3": "lcr_aspect_3",
        "gds_origin_global": "gds_origin",
        "pH(t)": "gds_ph", "pO2(t)": "gds_po2", "pCO2(t)": "gds_pco2",
        "Bicarbonates calculé": "gds_hco3", "SO2": "gds_so2"
    }

    # 6 - Filter and pivot
    df_filtered = df_glims_raw[df_glims_raw["analysis"].isin(analysis_map.keys())].copy()

    df_pivot = df_filtered.pivot_table(
        index=["nda", "sample_date"],
        columns="analysis",
        values="result",
        aggfunc="first"
    ).reset_index()

    # 7 - Rename to English
    df_pivot.rename(columns=analysis_map, inplace=True)
    df_pivot.columns.name = None

    return df_pivot

# #######################################################
# C) MAIN
# #######################################################
def main():
    print(f"\n=== Loading GLIMS files ({YEAR_START}-{YEAR_END}) ===")
    df_glims_raw = append_all_glims()

    if df_glims_raw.empty:
        print("No data loaded — exiting.")
        return pd.DataFrame()

    print("\n=== Pivot processing ===")
    df_glims = process_glims_pivot(df_glims_raw)

    # Keep earliest sample per patient
    df_glims.sort_values(by=["nda", "sample_date"], inplace=True)
    df_final = df_glims.groupby("nda", as_index=False).first()

    print(f"\n✅ Done: {df_final.shape[0]} unique patients.")
    return df_final

df_final = main()

# --- Summary ---
print("\n--- FINAL COLUMNS ---")
print(df_final.columns.tolist())

filling_rate = (df_final.notna().mean() * 100).round(2)
df_summary = filling_rate.to_frame(name="Filling Rate (%)").sort_values("Filling Rate (%)", ascending=False)
print("\n--- FILLING RATE SUMMARY ---")
print(df_summary)

# --- Save ---
df_final.to_csv(output_csv, index=False)
print(f"\n✅ Saved to: {output_csv}")

### 2022 ###

In [ ]:
# import os
# import glob
# import pandas as pd
# import numpy as np
#
# # #######################################################
# # # C) APPEND ALL GLIMS FILES (RAW)
# # #######################################################
# def append_all_glims():
#     glims_path = "../Datanad/subset_2022_PEL"
#     all_glims_files = glob.glob(os.path.join(glims_path, "GLI*.xlsx"))
#     glims_files = [f for f in all_glims_files if not os.path.basename(f).startswith("~$")]
#
#     if not glims_files:
#         print("[GLIMS] No files found.")
#         return pd.DataFrame()
#
#     dfs = []
#     for f in glims_files:
#         try:
#             # Titles are on the second row (header=1)
#             df = pd.read_excel(f, sheet_name="Analyses", header=1, dtype={"N° venue": str})
#
#             # --- RAW VALUE BACKUP ---
#             raw_with_dot = df["Résultat brute de l'analyse"].astype(str).str.replace(',', '.', regex=False)
#
#             # --- MERGE SOURCES ---
#             df['mixed_result'] = df["Résultat de l'analyse"].astype(str).replace('nan', np.nan)
#             df['mixed_result'] = df['mixed_result'].fillna(raw_with_dot)
#
#             dfs.append(df)
#         except Exception as e:
#             print(f"[GLIMS] Error reading {f}: {e}")
#
#     if not dfs:
#         return pd.DataFrame()
#
#     df_glims_raw = pd.concat(dfs, ignore_index=True)
#     print(f"[GLIMS raw] => {df_glims_raw.shape[0]} rows.")
#     return df_glims_raw
#
# # #######################################################
# # # D) PROCESS GLIMS (PIVOT)
# # #######################################################
# def process_glims_pivot(df_glims_raw):
#     # 1 - Rename columns for simplicity
#     rename_map = {
#         "N° venue": "nda",
#         "Date Prélèvement (en date)": "sample_date",
#         "Libellé analyse détaillée": "analysis",
#         "mixed_result": "result",
#     }
#     df_glims_raw.rename(columns=rename_map, inplace=True, errors="ignore")
#
#     # 2 - Clean patient IDs and dates
#     if "nda" in df_glims_raw.columns:
#         df_glims_raw["nda"] = df_glims_raw["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#     if "sample_date" in df_glims_raw.columns:
#         df_glims_raw["sample_date"] = pd.to_datetime(df_glims_raw["sample_date"], errors="coerce")
#
#     # 3 - Normalize analysis labels
#     df_glims_raw["analysis"] = df_glims_raw["analysis"].astype(str).str.replace(r'\xa0', ' ', regex=True).str.strip()
#     analysis_lower = df_glims_raw["analysis"].str.lower()
#
#     # --- FORCE EXPLICIT NAMES ---
#     # a. Culture
#     mask_culture = df_glims_raw["analysis"] == "Culture"
#     df_glims_raw.loc[mask_culture, "analysis"] = "culture_global"
#
#     # b. GDS Origin
#     mask_gds = analysis_lower.str.contains("origine", na=False) & analysis_lower.str.contains("gds", na=False)
#     df_glims_raw.loc[mask_gds, "analysis"] = "gds_origin_global"
#
#     # c. LCR (Cerebrospinal fluid) aspects mapping
#     lcr_mapping = {
#         "Aspect du LCR": "lcr_aspect_1",
#         "Asp.LCR centrifugé": "lcr_aspect_2",
#         "Aspect": "lcr_aspect_3"
#     }
#     df_glims_raw["analysis"] = df_glims_raw["analysis"].replace(lcr_mapping)
#
#     # 4 - Clean result values (vectorized)
#     def clean_values_vectorized(df):
#         """Vectorized cleaning of GLIMS results"""
#         result = df['result'].astype(str).str.strip()
#         analysis = df['analysis'].astype(str)
#
#         # Masks
#         mask_culture = analysis == "culture_global"
#         mask_lcr = analysis.isin(["lcr_aspect_1", "lcr_aspect_2", "lcr_aspect_3"])
#         mask_gds = analysis == "gds_origin_global"
#         mask_empty = result.str.lower().isin(["", "nan", "none"])
#         mask_nren = result.str.contains(r'\{<NREN', na=False, regex=True)
#
#         cleaned = result.copy()
#
#         # a. CULTURE
#         cleaned.loc[mask_culture & mask_empty] = np.nan
#         cleaned.loc[mask_culture & ~mask_empty & ~mask_nren] = "YES"
#         cleaned.loc[mask_culture & mask_nren] = "NREN"
#
#         # b. EMPTY VALUES (except culture)
#         cleaned.loc[mask_empty & ~mask_culture] = np.nan
#
#         # c. GDS - Clean except NREN
#         gds_mask_clean = mask_gds & ~mask_nren
#         if gds_mask_clean.any():
#             gds_cleaned = (cleaned.loc[gds_mask_clean]
#                            .str.replace('{', '', regex=False)
#                            .str.replace('}', '', regex=False)
#                            .str.replace('<', '', regex=False)
#                            .str.replace('BC_', '', regex=False)
#                            .str.strip())
#             cleaned.loc[gds_mask_clean] = gds_cleaned
#         cleaned.loc[mask_gds & mask_nren] = "NREN"
#
#         # d. NUMERIC - extract numbers
#         mask_numeric = ~(mask_culture | mask_lcr | mask_gds | mask_empty | mask_nren)
#         if mask_numeric.any():
#             numeric_extracted = cleaned.loc[mask_numeric].str.extract(r'(\d+\.?\d*)', expand=False)
#             cleaned.loc[mask_numeric] = numeric_extracted.fillna(cleaned.loc[mask_numeric])
#
#         # e. NUMERIC NREN
#         cleaned.loc[mask_nren & ~(mask_culture | mask_lcr | mask_gds)] = "NREN"
#
#         return cleaned
#
#     df_glims_raw["result"] = clean_values_vectorized(df_glims_raw)
#
#     # 5 - Final mapping of explicit column names
#     analysis_map = {
#         "Créatinine sg": "creatinine", "Urée sg": "urea",
#         "Potassium sg": "potassium", "Sodium sg": "sodium", "Calcium sg arsenazo": "calcium",
#         "Troponine I HS": "troponine", "CKMB": "ckmb", "BNP": "bnp",
#         "ASAT (TGO)": "asat", "ALAT (TGP)": "alat", "Bilirubine totale": "bili_total",
#         "PAL sg": "alp", "Lipase sg": "lipase",
#         "Leucocytes": "leucocytes", "PNeutro Va": "neutrophils", "Lympho Va": "lymphocytes",
#         "Monocytes Va": "monocytes", "PBaso Va": "basophils", "PEosino Va": "eosinophils",
#         "Hémoglobine": "hemoglobine", "PlaquettesEDTA": "platelets",
#         "TP Taux Prothrombine": "pt", "TCA patient": "aptt", "FibrinogèneClauss": "fibrinogen",
#         "Activité AXa (HNF)": "axa_hnf", "Activité AXa Xarelto": "axa_xarelto",
#         "Activité AXa Eliquis": "axa_eliquis", "Activité AXa HBPM": "axa_hbpm",
#         "Anti-IIa Pradaxa": "aiia_pradaxa", "Activité AXa Arixtra": "axa_arixtra",
#         "Activité AXa Orgaran": "axa_orgaran",
#         "CK sg": "ck", "D-Dimères dosage": "ddimer",
#         "Lactate": "lactates_a", "Lactate plasma vein": "lactates_v", "Calcium ionisé": "calcium_ionized",
#         "Procalcitonine ser": "pct", "CRP": "crp",
#         "Fer sg": "iron", "Ferritine": "ferritin",
#         "culture_global": "culture",
#         "lcr_aspect_1": "lcr_aspect_1", "lcr_aspect_2": "lcr_aspect_2", "lcr_aspect_3": "lcr_aspect_3",
#         "gds_origin_global": "gds_origin",
#         "pH(t)": "gds_ph", "pO2(t)": "gds_po2", "pCO2(t)": "gds_pco2",
#         "Bicarbonates calculé": "gds_hco3", "SO2": "gds_so2"
#     }
#
#     # 6 - Filter and pivot
#     df_filtered = df_glims_raw[df_glims_raw["analysis"].isin(analysis_map.keys())].copy()
#
#     df_pivot = df_filtered.pivot_table(
#         index=["nda", "sample_date"],
#         columns="analysis",
#         values="result",
#         aggfunc="first"  # keep first sample if duplicates
#     ).reset_index()
#
#     # 7 - Final rename
#     df_pivot.rename(columns=analysis_map, inplace=True)
#
#     return df_pivot
#
# # #######################################################
# # # MAIN
# # #######################################################
# def main():
#     print("\n=== Reading GLIMS (Merge & Decimal Backup) ===")
#     df_glims_raw = append_all_glims()
#
#     if df_glims_raw.empty:
#         return pd.DataFrame()
#
#     print("\n=== Pivot Processing & Explicit Columns ===")
#     df_glims = process_glims_pivot(df_glims_raw)
#
#     # Merge by patient_id, keep earliest sample per patient
#     df_glims.sort_values(by=["nda", "sample_date"], inplace=True)
#     df_final = df_glims.groupby("nda", as_index=False).first()
#
#     print(f"✅ Done: {df_final.shape[0]} unique patients with lab columns.")
#     return df_final
#
# if __name__ == "__main__":
#     df_final = main()

In [ ]:
# # 1. Column list
# print("--- FINAL COLUMNS LIST ---")
# print(df_final.columns.tolist())
#
# # 2. computing percentage of non-null values per column to have an idea of the completeness of each lab variable
# remplissage = (df_final.notna().mean() * 100).round(2)
#
# # making it a df for better visualization and sorting it from the most complete to the least complete
# df_bilan = remplissage.to_frame(name='Filling Rate (%)')
# df_bilan = df_bilan.sort_values(by='Filling Rate (%)', ascending=False)
#
# print("\n--- FILLING SUMMARY  ---")
# print(df_bilan)

In [ ]:
# # Saving in csv
# output_name = "df_bio_subset22pelglims.csv"
# (df_final.to_csv(output_name, index=False))
# print(f"✅ File saved under : {output_name}")

In [ ]:
# import os
# import glob
# import pandas as pd
# import numpy as np
#
# # #######################################################
# # # C) APPEND TOUS LES FICHIERS GLIMS (brut)
# # #######################################################
# def append_all_glims():
#     glims_path = "../Datanad/subset_2022_PEL"
#     all_glims_files = glob.glob(os.path.join(glims_path, "GLI*.xlsx"))
#     glims_files = [f for f in all_glims_files if not os.path.basename(f).startswith("~$")]
#
#     if not glims_files:
#         print("[Glims] Aucun fichier trouvé.")
#         return pd.DataFrame()
#
#     dfs = []
#     for f in glims_files:
#         try:
#             # On utilise header=1 car tes titres sont en ligne 2 de l'Excel
#             df = pd.read_excel(f, sheet_name="Analyses", header=1, dtype={"N° venue": str})
#
#             # --- LE BACK-UP VIRGULE ---
#             # On prépare la colonne brute en remplaçant la virgule par un point pour ne pas perdre les décimales
#             brute_au_point = df["Résultat brute de l'analyse"].astype(str).str.replace(',', '.', regex=False)
#
#             # --- LA FUSION DES SOURCES ---
#             # Priorité à la colonne "Résultat de l'analyse" (formatée au point par Glims)
#             # Si elle est vide (NaN), on prend notre secours 'brute_au_point'
#             df['res_bio_mixte'] = df["Résultat de l'analyse"].astype(str).replace('nan', np.nan)
#             df['res_bio_mixte'] = df['res_bio_mixte'].fillna(brute_au_point)
#
#             dfs.append(df)
#         except Exception as e:
#             print(f"[Glims] Erreur lecture {f} : {e}")
#
#     if not dfs:
#         return pd.DataFrame()
#
#     df_glims_raw = pd.concat(dfs, ignore_index=True)
#     print(f"[Glims raw] => {df_glims_raw.shape[0]} lignes.")
#     return df_glims_raw
#
# # #######################################################
# # # D) TRAITEMENT GLIMS (pivot)
# # #######################################################
# def process_glims_pivot(df_glims_raw):
#     # 1 - Renommage pour simplifier la manipulation
#     # On utilise notre nouvelle colonne 'res_bio_mixte' comme source de valeur
#     rename_base = {
#         "N° venue": "nda",
#         "Date Prélèvement (en date)": "date_prlvt",
#         "Libellé analyse détaillée": "analyse",
#         "res_bio_mixte": "resultat",
#     }
#     df_glims_raw.rename(columns=rename_base, inplace=True, errors="ignore")
#
#     # 2 - Nettoyage NDA et Dates
#     if "nda" in df_glims_raw.columns:
#         df_glims_raw["nda"] = df_glims_raw["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#     if "date_prlvt" in df_glims_raw.columns:
#         df_glims_raw["date_prlvt"] = pd.to_datetime(df_glims_raw["date_prlvt"], errors="coerce")
#
#     # 3 - NORMALISATION DES LIBELLÉS
#     df_glims_raw["analyse"] = df_glims_raw["analyse"].astype(str).str.replace(r'\xa0', ' ', regex=True).str.strip()
#     ana_lower = df_glims_raw["analyse"].str.lower()
#
#     # --- FORÇAGE DES NOMS (Explicites) ---
#
#
#     # a. Culture : Seulement si le texte est EXACTEMENT "Culture"
#     # (Tes résultats ### seront gérés dans la fonction clean_values)
#     mask_culture = (df_glims_raw["analyse"] == "Culture")
#     df_glims_raw.loc[mask_culture, "analyse"] = "culture_global"
#
#     # b. GDS Origine : On garde le contient car il y a souvent des espaces ou variantes
#     mask_gds = ana_lower.str.contains("origine", na=False) & ana_lower.str.contains("gds", na=False)
#     df_glims_raw.loc[mask_gds, "analyse"] = "gds_origine_global"
#
#     # c. LCR : Mapping précis pour les 3 variantes que tu as citées
#     # lcr_mapping = {
#     #     "Aspect du LCR": "lcr_aspect_global",
#     #     "Asp.LCR centrifugé": "lcr_aspect_global"
#     # }
#     # On applique le mapping. Les valeurs non trouvées restent inchangées (na=False n'est pas nécessaire ici)
#     #df_glims_raw["analyse"] = df_glims_raw["analyse"].replace(lcr_mapping)
#
#     # d. Lactates : Remplacement spécifique pour tes deux variantes
#     # lactate_mapping = {
#     #     "Lactate": "lactates_global",
#     #     "Lactate plasma vein": "lactates_global"
#     # }
#     #df_glims_raw["analyse"] = df_glims_raw["analyse"].replace(lactate_mapping)
#
#
#
#
#     # 4 - Nettoyeur de valeurs (On garde le texte pour l'aspect et nettoie le numérique)
#     # def clean_values(row):
#     #     """
#     #     Nettoie les résultats Glims selon le type d'analyse.
#     #     - Culture -> Devient 'OUI' si présence de données (même ###)
#     #     - GDS -> Supprime les balises techniques {<BC_...}
#     #     - LCR -> Garde le texte propre (Eau de roche)
#     #     - Numérique -> Extrait le nombre pur (gère les unités et les points)
#     #     """
#     #     ana = str(row['analyse'])
#     #     val = str(row['resultat']).strip()
#     #
#     # # a. SÉCURITÉ CULTURE (On transforme ### ou autre en 'OUI')
#     #     if ana == "culture_global":
#     #         if val.lower() in ["nan", "none", ""]:
#     #             return np.nan
#     #         else:
#     #             return "OUI"
#     #
#     #     # b. SÉCURITÉ DE BASE (Vrais vides)
#     #     if val.lower() in ["", "nan", "none"]:
#     #         return np.nan
#     #
#     #     # c. CAS QUALITATIFS (GDS)
#     #     # GDS : On vire le formatage technique {<BC_...}
#     #     if ana == "gds_origine_global":
#     #         return (val.replace('{', '')
#     #                    .replace('}', '')
#     #                    .replace('<', '')
#     #                    .replace('BC_', '')
#     #                    .strip())
#     #
#     #     # d. CAS NUMÉRIQUES (Glucose, Lactates, etc.)
#     #     # On tente d'extraire le nombre (ex: "5.2 mmol/L" -> "5.2")
#     #     # On s'appuie sur le fait qu'on a déjà forcé le point (.) au lieu de la virgule (,)
#     #     match = pd.Series(val).str.extract(r'(\d+\.?\d*)')[0]
#     #
#     #     if not match.dropna().empty:
#     #         return match.iloc[0]
#     #
#     #     # Si aucun chiffre n'est trouvé dans une colonne numérique (ex: "Hémolysé")
#     #     # on renvoie la valeur telle quelle pour ne pas perdre l'info avant le GMM
#     #     return val
#     #
#     # df_glims_raw["resultat"] = df_glims_raw.apply(clean_values, axis=1)
#
#     def clean_values_vectorized(df):
#         """Version vectorisée sans transformer NREN en NaN"""
#
#         resultat = df['resultat'].astype(str).str.strip()
#         analyse = df['analyse'].astype(str)
#
#         # ✅ MASQUES CORRECTEMENT DÉFINIS
#         mask_culture = (analyse == "culture_global")
#         mask_lcr = analyse.isin(["lcr_aspect_1", "lcr_aspect_2", "lcr_aspect_3"])  # ✅ .isin() !
#         mask_gds = (analyse == "gds_origine_global")
#         mask_vide = resultat.str.lower().isin(["", "nan", "none"])
#         mask_nren = resultat.str.contains(r'\{<NREN', na=False, regex=True)
#
#         cleaned = resultat.copy()
#
#         # a. CULTURE
#         cleaned.loc[mask_culture & mask_vide] = np.nan
#         cleaned.loc[mask_culture & ~mask_vide & ~mask_nren] = "OUI"
#         cleaned.loc[mask_culture & mask_nren] = "NREN"
#
#         # b. VALEURS VIDES (sauf culture)
#         cleaned.loc[mask_vide & ~mask_culture] = np.nan
#
#         # c. GDS - Nettoyer SAUF si NREN
#         gds_mask_clean = mask_gds & ~mask_nren
#         if gds_mask_clean.any():  # ✅ Sécurité si aucune ligne GDS
#             gds_cleaned = (cleaned.loc[gds_mask_clean]
#                            .str.replace('{', '', regex=False)
#                            .str.replace('}', '', regex=False)
#                            .str.replace('<', '', regex=False)
#                            .str.replace('BC_', '', regex=False)
#                            .str.strip())
#             cleaned.loc[gds_mask_clean] = gds_cleaned
#
#         cleaned.loc[mask_gds & mask_nren] = "NREN"
#
#         # d. NUMÉRIQUES - Extraire le nombre (sauf NREN)
#         mask_numerique = ~(mask_culture | mask_lcr | mask_gds | mask_vide | mask_nren)
#
#         if mask_numerique.any():  # ✅ Sécurité si aucune valeur numérique
#             numerique_extrait = cleaned.loc[mask_numerique].str.extract(r'(\d+\.?\d*)', expand=False)
#             cleaned.loc[mask_numerique] = numerique_extrait.fillna(cleaned.loc[mask_numerique])
#
#         # Pour les numériques avec NREN, on garde "NREN"
#         cleaned.loc[mask_nren & ~(mask_culture | mask_lcr | mask_gds)] = "NREN"
#
#         return cleaned
#
#     df_glims_raw["resultat"] = clean_values_vectorized(df_glims_raw)
#
#     # 5 - Dictionnaire de Mapping Final (Colonnes Explicites)
#     analyses_map = {
#         "Créatinine sg": "creat", "Urée sg": "urea",
#         "Potassium sg": "potassium", "Sodium sg": "sodium", "Calcium sg arsenazo": "calcium",
#         "Troponine I HS": "tropo", "CKMB": "ckmb", "BNP": "bnp",
#         "ASAT (TGO)": "asat", "ALAT (TGP)": "alat", "Bilirubine totale": "bili_tot",
#         "PAL sg": "alp", "Lipase sg": "lipase",
#         "Leucocytes": "leuco", "PNeutro Va": "pnn", "Lympho Va": "lymphocytes",
#         "Monocytes Va": "monocytes", "PBaso Va": "pnb", "PEosino Va": "pne",
#         "Hémoglobine": "hb", "PlaquettesEDTA": "plqt",
#         "TP Taux Prothrombine": "tp", "TCA patient": "tca", "FibrinogèneClauss": "fibrinogene",
#         "Activité AXa (HNF)": "axa_hnf", "Activité AXa Xarelto": "axa_xarelto",
#         "Activité AXa Eliquis": "axa_eliquis", "Activité AXa HBPM": "axa_hbpm",
#         "Anti-IIa Pradaxa": "aiia_pradaxa", "Activité AXa Arixtra": "axa_arixtra",
#         "Activité AXa Orgaran": "axa_orgaran",
#         "CK sg": "ck", "D-Dimères dosage": "ddimere",
#         "Lactate": "lactates_a",
#         "Lactate plasma vein": "lactates_v", "Calcium ionisé": "calcium_ionise",
#         "Procalcitonine ser": "pct", "CRP": "crp",
#         "Fer sg": "fer", "Ferritine": "ferritine",
#         "culture_global": "culture",
#         "Aspect du LCR": "lcr_aspect_1",
#         "Asp.LCR centrifugé": "lcr_aspect_2",
#         "Aspect": "lcr_aspect_3",
#         "gds_origine_global": "gds_origine",
#         "pH(t)": "gds_ph", "pO2(t)": "gds_po2", "pCO2(t)": "gds_pco2",
#         "Bicarbonates calculé": "gds_hco3", "SO2": "gds_so2"
#
#     }
#
#     # 6 - Filtrage et Pivot
#     df_g = df_glims_raw[df_glims_raw["analyse"].isin(analyses_map.keys())].copy()
#
#     df_pivot = df_g.pivot_table(
#         index=["nda", "date_prlvt"],
#         columns="analyse",
#         values="resultat",
#         aggfunc="first" # Prend le premier prélèvement si doublons
#     ).reset_index()
#
#     # 7 - Renommage final
#     df_pivot.rename(columns=analyses_map, inplace=True)
#
#     return df_pivot
#
# # #######################################################
# # # MAIN
# # #######################################################
# def main():
#     print("\n=== Lecture Glims (Fusion & Backup Décimales) ===")
#     df_glims_raw = append_all_glims()
#
#     if df_glims_raw.empty:
#         return pd.DataFrame()
#
#     print("\n=== Traitement Pivot & Colonnes Explicites ===")
#     df_glims = process_glims_pivot(df_glims_raw)
#
#     # Fusion par NDA (on garde le premier prélèvement chronologique)
#     df_glims.sort_values(by=["nda", "date_prlvt"], inplace=True)
#     df_bio_filled = df_glims.groupby("nda", as_index=False).first()
#
#     print(f"✅ Terminé : {df_bio_filled.shape[0]} patients uniques avec colonnes LABO.")
#
#     return df_bio_filled
#
#
#
# if __name__ == "__main__":
#     df_final = main()

In [ ]:
# # 1. Liste simple des colonnes
# print("--- LISTE DES COLONNES FINALES ---")
# print(df_final.columns.tolist())
#
# # 2. Calcul du pourcentage de remplissage (Taux de complétude)
# # On prend la moyenne des valeurs non-nulles et on multiplie par 100
# remplissage = (df_final.notna().mean() * 100).round(2)
#
# # On transforme ça en DataFrame pour que ce soit plus joli (avec un tri)
# df_bilan = remplissage.to_frame(name='Taux de remplissage (%)')
# df_bilan = df_bilan.sort_values(by='Taux de remplissage (%)', ascending=False)
#
# print("\n--- BILAN DE COMPLÉTUDE (du plus plein au plus vide) ---")
# print(df_bilan)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# #######################################################
# # A) APPEND TOUS LES FICHIERS SYNERGY (brut)
# #######################################################
# def append_all_synergy():
#     #synergy_path = "../Datanad/Bio/synergy"
#     synergy_path = "../Datanad/subset_2022_PEL"
#     all_synergy_files = glob.glob(os.path.join(synergy_path, "CGJ 071*.xlsx"))
#     # Utilisez ceci pour filtrer les fichiers temporaires (~$)
#     synergy_files = [f for f in all_synergy_files if not os.path.basename(f).startswith("~$")]
#
#     if not synergy_files:
#         print("[Synergy] Aucun fichier trouvé.")
#         return pd.DataFrame()
#
#     dfs = []
#     for f in synergy_files:
#         try:
#             # Lecture brut, skiprows=1 si nécessaire
#             df = pd.read_excel(f, sheet_name="Nombre", skiprows=1)
#             dfs.append(df)
#         except Exception as e:
#             print(f"[Synergy] Erreur lecture {f} : {e}")
#
#     if not dfs:
#         print("[Synergy] Aucun DF concaténé.")
#         return pd.DataFrame()
#
#     df_syn_raw = pd.concat(dfs, ignore_index=True)
#     print(f"[Synergy raw] => {df_syn_raw.shape[0]} lignes, {df_syn_raw.shape[1]} colonnes.")
#     return df_syn_raw

#######################################################
# B) TRAITEMENT SYNERGY => METTRE A JOUR LES BIO QUE JAI DEJA RAJOUTE SUR GLIMS
#######################################################
# def process_synergy(df_syn_raw):
#     """
#     On suppose que Synergy est déjà au format large :
#     - On renomme les colonnes
#     - On conserve seulement les colonnes cibles
#     - On convertit les dates
#     """
#     # Dictionnaire de renommage Synergy
#     SYNERGY_RENAME = {
#         "Unnamed: 1": "nda",
#         "Unnamed: 2": "date",
#         "Unnamed: 3": "date_prlvt",
#         "N° venue": "nda",
#         "Date Prélèvement (en date)": "date_bio",
#
#         "C1CRS CREATININE sg": "creat",
#         "C1URS UREE sg " : "uree" # rajout nad
#         "C1KS POTASSIUM sg": "potassium",
#         "C1NAS SODIUM sg": "sodium",
#         "C1CAB CALCIUM sg arsenazo": "calcium",
#         "C3TNI TROPONINE I": "tropo",
#         "CCKMB -CREATININE KINASE M": "ckmb", # rajout nad
#         "C3BBN BNP DXI": "bnp", # rajout nad
#         "C2CRP CRP": "crp",
#         "C2TGO ASAT(TGO)": "asat",
#         "C2TGP ALAT(TGP)": "alat",
#         "C1BTS BILIRUBINE totale": "bili_tot",
#         ""C2PAS PAL sg" : "alp", # rajout nad
#         "C2LIS LIPASE sg": "lipase",
#         "HGB Leucocytes": "leuco",
#         "HFSNV PN valeur absolue": "neutro",
#         "HFSMV Mono valeur absolue": "monocytes",
#         "HFSLV Lymp valeur absolue": "lympho",
#         "HFSB  Basophiles": "basophiles",
#         "HFSE  Eosinophiles": "eosinophiles",
#         "HHB  Hémoglobine": "hemoglobine",
#         "HVGM  VGM": "vgm",
#         "HPLAQ  Plaquettes": "plaq",
#         "HTPS Taux Prothrombine": "tp",
#         "HTCAP *TCA patient" : "tca", # rajout nad
#         "HFGD1 FibrinogèneDérivéTP" : "fibrinogene", # rajout nad (pas le meme que dans glims ou c'est avec technique clauss)
#         "HELQ *ActivitéAXa ELIQUIS" : "axa_eliquis",
#         "HAXAS *Activité AXa HépSt" : "axa_hnf",
#         "HAXAB *Activité AXa HBPM" : "axa_hbpm",
#         "HDDEP  DD ELISA/ T": "ddimere",
#         "C2CPK CK sg": "ck", # rajout nad
#     }
#
#     # Liste de colonnes cibles
#     SYNERGY_COLS = [
#         "nda", "date", "date_prlvt", "date_bio",
#         "creat", "potassium", "sodium", "calcium",
#         "tropo", "crp", "asat", "alat", "bili_tot",
#         "lipase", "leuco", "neutro", "monocytes",
#         "lympho", "basophiles", "eosinophiles",
#         "hemoglobine", "vgm", "plaq", "TP", "inr",
#         "ddimere"
#     ]
#
#     # Renommage
#     df_syn_raw.rename(columns=SYNERGY_RENAME, inplace=True, errors="ignore")
#
#     # Conversion nda en string
#     if "nda" in df_syn_raw.columns:
#         df_syn_raw["nda"] = df_syn_raw["nda"].astype(str)
#
#     # Convertir date_prlvt / date_bio / date
#     for date_col in ["date_prlvt", "date_bio", "date"]:
#         if date_col in df_syn_raw.columns:
#             df_syn_raw[date_col] = pd.to_datetime(df_syn_raw[date_col], errors="coerce", dayfirst=True)
#
#     # Filtrer
#     keep = [c for c in SYNERGY_COLS if c in df_syn_raw.columns]
#     df_syn = df_syn_raw[keep].copy()
#
#     print(f"[Synergy processed] => {df_syn.shape[0]} lignes, {df_syn.shape[1]} colonnes.")
#     return df_syn

#######################################################
# C) APPEND TOUS LES FICHIERS GLIMS (brut)
#######################################################





#######################################################
# D) TRAITEMENT GLIMS (pivot)
#######################################################















#######################################################
# E) Fonction de remplissage => 1 date par nda
#######################################################
# def fill_from_subsequent_rows(group):
#     """
#     group = data pour un nda, trié par date_prlvt asc.
#     On part de la première date, et on remplit les colonnes NaN
#     avec les valeurs trouvées dans les dates ultérieures.
#     """
#     if len(group) == 1:
#         return group.iloc[0]
#
#     filled = group.iloc[0].copy()
#     for i in range(1, len(group)):
#         row = group.iloc[i]
#         for col in group.columns:
#             if col == "date_prlvt":
#                 continue  # on garde la plus ancienne date
#             if pd.isna(filled[col]) and not pd.isna(row[col]):
#                 filled[col] = row[col]
#     return filled





#######################################################
# MAIN   A CORRIGER ET RAJOUTER SYNERGY
#######################################################
# def main():
#     # 1) Synergy
#     print("=== Lecture Synergy (append brut) ===")
#     df_syn_raw = append_all_synergy()
#     print("\n=== Traitement Synergy ===")
#     df_syn = process_synergy(df_syn_raw)
#
#     # 2) Glims
#     print("\n=== Lecture Glims (append brut) ===")
#     df_glims_raw = append_all_glims()
#     print("\n=== Traitement Glims (pivot) ===")
#     df_glims = process_glims_pivot(df_glims_raw)
#
#     # 3) Append vertical
#     print("\n=== Concat Synergy + Glims ===")
#     df_bio = pd.concat([df_syn, df_glims], ignore_index=True)
#     print(f"[Append synergy+glims] => {df_bio.shape[0]} lignes, {df_bio.shape[1]} colonnes.")
#
#     # 4) Tri par nda, date_prlvt
#     if "date_prlvt" in df_bio.columns:
#         df_bio["date_prlvt"] = pd.to_datetime(df_bio["date_prlvt"], errors="coerce")
#         df_bio.sort_values(by=["nda", "date_prlvt"], inplace=True)
#
#     # 5) fill => 1 date par nda, on remplit
#     print("\n=== Remplissage => 1 date par nda ===")
#     df_bio_filled = df_bio.groupby("nda", as_index=False).apply(fill_from_subsequent_rows)
#     df_bio_filled.reset_index(drop=True, inplace=True)
#
#     print(f"[df_bio_filled] => {df_bio_filled.shape[0]} lignes, {df_bio_filled.shape[1]} colonnes.")
#     print(df_bio_filled.head(20))
#     return df_bio_filled
#
# if __name__ == "__main__":
#     df_final = main()
#     print("\n--- Script terminé. ---")



Bon je ne sais pas encore ce que e fais des paitents aui ont plusieurs antiXa differnent, on verra plus tard

# 6. Administrative Data #

## a. 2022 to 2025 ##

In [ ]:
# ================================================================
# CODE FOR CSV FILE
#=========================================================================

# --- Config ---
DATA_PATH = "../Datanad/subset_data"
OUTPUT_PATH = "Datasets"
YEAR_START = 2022
YEAR_END = 2025
output_csv = os.path.join(OUTPUT_PATH, f"df_admin_{YEAR_START}_{YEAR_END}.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Load CSV ---
csv_file = os.path.join(DATA_PATH, "CGJ_084_-_IMS_202225.csv")
df_final = pd.read_csv(csv_file, sep=";", encoding="utf-8-sig")
print(f"✅ Loaded: {len(df_final)} rows")
print("Colonnes trouvées :", df_final.columns.tolist())

# --- Rename ---
df_final.rename(columns={
    "UG entrée séjour Code":            "uam_service",
    "Nda":                              "nda",
    "Date entrée UG entrée séjour":     "date_entree_urg",
    "Date sortie UG entrée séjour":     "date_sortie_urg",
    "Date Sortie Venue":                "date_sortie_chu",
    "Urgence Date Sortie Completee":    "date_sortie_urg_completee",
    "Nom du patient":                   "nom",
    "Prénom du patient":                "prenom",
    "Date de naissance du patient":     "date_naissance",
    "Libelle de la ville de naissance": "ville_naissance",
    "Pays Naissance":                   "pays_naissance",
    "Adresse du patient (rue)":         "adresse_rue",
    "Code postal de la ville":          "adresse_cp",
    "Code commune de la ville":         "adresse_insee",
    "Libelle de la ville":              "adresse_ville",
    "Sejour Mode Sortie Libelle":       "mode_sortie_chu",
    "Décision urgence Libellé":         "decision_urgence",
}, inplace=True, errors="ignore")

# --- Filter UAM 9780 ---
n_before = len(df_final)
df_final = df_final[df_final["uam_service"].astype(str).str.strip() == "9780"].copy()
print(f"UAM filter: kept {len(df_final)} rows (removed {n_before - len(df_final)}).")

# --- Cleaning ---
df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_final["date_entree_urg"] = pd.to_datetime(df_final["date_entree_urg"],
                                          format="%Y/%m/%d %H:%M:%S", errors="coerce")

n_before = len(df_final)
df_final = df_final[
    (df_final["date_entree_urg"].dt.year >= YEAR_START) &
    (df_final["date_entree_urg"].dt.year <= YEAR_END)
].copy()
print(f"Year filter: removed {n_before - len(df_final)} rows outside {YEAR_START}-{YEAR_END}.")

# --- Dédoublonnage ---
df_final["count_nonnull"] = df_final.notna().sum(axis=1)
df_final.sort_values(by="count_nonnull", ascending=False, inplace=True)
n_before = len(df_final)
df_final.drop_duplicates(subset="nda", keep="first", inplace=True)
print(f"{n_before - len(df_final)} duplicates removed.")

# --- Sélection colonnes finales ---
cols_to_keep = {
    "uam_service":      "uam_service",
    "nda":              "nda",
    "date_naissance":   "date_naissance",
    "date_entree_urg":      "date_entree_urg",
    "date_sortie_urg":      "date_sortie_urg",
    "date_sortie_urg_completee": "date_sortie_urg_completee",
    "decision_urgence": "decision_urgence",
    "date_sortie_chu":      "date_sortie_chu",
    "mode_sortie_chu":      "mode_sortie_chu",
}

existing_cols = [c for c in cols_to_keep.keys() if c in df_final.columns]
df_admin = df_final[existing_cols].copy()
df_admin.sort_values(by="date_entree_urg", ascending=True, inplace=True)

print(f"\n✅ Done: {len(df_admin)} unique patients, {len(df_admin.columns)} colonnes.")

# --- Export ---
df_admin.to_csv(output_csv, index=False)
print(f"✅ Saved to: {output_csv}")

In [ ]:
print("mode_sortie_chu" in df_admin.columns)
print(df_admin["mode_sortie_chu"].value_counts(dropna=False).head(10))

In [ ]:
# # ================================================================
# # CODE FOR EXCEL FILES
# #=========================================================================
#
#
#
#
# import os
# import glob
# import pandas as pd
# import numpy as np
#
# # --- Config ---
# DATA_PATH = "../Datanad/subset_data"
# OUTPUT_PATH = "Datasets"
# YEAR_START = 2022
# YEAR_END = 2025
# output_csv = os.path.join(OUTPUT_PATH, f"df_admin_{YEAR_START}_{YEAR_END}.csv")
#
# os.makedirs(OUTPUT_PATH, exist_ok=True)
#
# ######################################
# # Part 1 : CGJ 084 files only
# ######################################
# pattern_084 = os.path.join(DATA_PATH, "CGJ_084_*.xlsx")
# files_084 = [f for f in glob.glob(pattern_084) if not os.path.basename(f).startswith("~$")]
#
# print(f"[084] {len(files_084)} file(s) found.")
#
# dfs_084 = []
#
#
# for i, f in enumerate(files_084):
#     try:
#         print(f"  -> Reading ({i+1}/{len(files_084)}): {os.path.basename(f)}...")
#         df = pd.read_excel(f)
#         df["source_file"] = os.path.basename(f)  # ← ajoute cette ligne
#         dfs_084.append(df)
#     except Exception as e:
#         print(f"  ❌ Error on {os.path.basename(f)}: {e}")
#
# if not dfs_084:
#     raise ValueError("No CGJ 084 files found — check path and pattern.")
#
# df_final = pd.concat(dfs_084, ignore_index=True)
# print("Colonnes trouvées :", df_final.columns.tolist())
# print(f"✅ Concatenation successful: {len(df_final)} raw rows.")
#
# df_final.rename(columns={
#     "UG entrée séjour Code":            "uam_service",
#     "Nda":                              "nda",
#     "Date entrée UG entrée séjour":     "date_entree",
#     "Date sortie UG entrée séjour":     "date_sortie",
#     "Nom du patient":                   "nom",
#     "Prénom du patient":                "prenom",
#     "Date de naissance du patient":     "date_naissance",
#     "Libelle de la ville de naissance": "ville_naissance",
#     "Pays Naissance":                   "pays_naissance",
#     "Numéro de securite sociale":       "nir",
#     "Adresse du patient (rue)":         "adresse_rue",
#     "Code postal de la ville":          "adresse_cp",
#     "Code commune de la ville":         "adresse_insee",
#     "Libelle de la ville":              "adresse_ville",
#     # --- AJOUTE CES DEUX LIGNES ICI ---
#     "Mode sortie séjour Libellé":       "mode_sortie",
#     "Décision urgence Libellé":         "decision_urgence",
# }, inplace=True, errors="ignore")
#
# # Filter UAM service 9780 only
# n_before = len(df_final)
# df_final = df_final[df_final["uam_service"].astype(str).str.strip() == "9780"].copy()
# print(f"  -> UAM filter: kept {len(df_final)} rows (removed {n_before - len(df_final)} from other services).")
#
#
#
# ######################################
# # Part 2 : Merge and clean
# ######################################
#
# if df_final.empty:
#     print("❌ Empty result — check file paths.")
# else:
#     # 1. Nettoyage du NDA
#     df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#
#     # 2. Conversion et filtrage des dates
#     if "date_entree" in df_final.columns:
#         df_final["date_entree"] = pd.to_datetime(df_final["date_entree"], errors="coerce")
#
#         print(f"  -> Filtering {YEAR_START}-{YEAR_END}...")
#         n_before = len(df_final)
#         df_final = df_final[
#             (df_final["date_entree"].dt.year >= YEAR_START) &
#             (df_final["date_entree"].dt.year <= YEAR_END)
#         ].copy()
#         print(f"  -> Removed {n_before - len(df_final)} rows outside range.")
#
#     # 3. Suppression des doublons (on garde la ligne la plus remplie)
#     df_final["count_nonnull"] = df_final.notna().sum(axis=1)
#     df_final.sort_values(by="count_nonnull", ascending=False, inplace=True)
#     n_before = len(df_final)
#     df_final.drop_duplicates(subset="nda", keep="first", inplace=True)
#     print(f"  -> {n_before - len(df_final)} duplicates removed.")
#
#     # 4. SÉLECTION ET RENOMMAGE DES COLONNES (C'est ici que ça change)
#     # On définit ce qu'on garde
#     cols_to_keep = {
#         "uam_service": "uam_service",
#         "nda": "nda",
#         "date_entree": "date_entree",
#         "date_sortie": "date_sortie",
#         "date_naissance": "date_naissance",
#         "mode_sortie": "mode_sortie",
#         "decision_urgence": "decision_urgence"
#     }
#
#     # On ne garde que celles qui existent vraiment dans le fichier
#     existing_cols = [c for c in cols_to_keep.keys() if c in df_final.columns]
#     df_admin = df_final[existing_cols].copy()
#
#     # On renomme les colonnes avec étoiles pour simplifier la suite
#     df_admin.rename(columns=cols_to_keep, inplace=True)
#
#     # 5. Tri final
#     if "date_entree" in df_admin.columns:
#         df_admin.sort_values(by="date_entree", ascending=True, inplace=True)
#
#     print(f"\n✅ Done: {len(df_admin)} unique patients avec {len(df_admin.columns)} colonnes.")
#
#     # --- Save ---
#     df_admin.to_csv(output_csv, index=False)
#     print(f"✅ Saved to: {output_csv}")
#
# # --- Création du subset 2022 spécifique ---
#     print("\n📦 Création du subset 2022...")
#
#     # On filtre sur l'année 2022
#     df_2022 = df_admin[df_admin["date_entree"].dt.year == 2022].copy()
#
#     # Nom du fichier (sans le dossier Dataset, donc à la racine de ton environnement)
#     subset_csv = "df_admin_subset22pel.csv"
#
#     df_2022.to_csv(subset_csv, index=False)
#
#     print(f"✅ Subset 2022 créé : {len(df_2022)} lignes.")
#     print(f"📍 Emplacement : {os.path.abspath(subset_csv)}")

In [ ]:
# import os
#
# DATA_PATH = "../Datanad/subset_data"
# all_files = os.listdir(DATA_PATH)
# hidden = [f for f in all_files if "084" in f]
#
# print("Tous les fichiers 084 (y compris cachés) :")
# for f in hidden:
#     print(f"  '{f}'")

### 2022

In [ ]:
# import os
# import glob
# import pandas as pd
# import numpy as np
#
# ######################################
# # Part 1 : CGJ 084 files
# ######################################
# path_084 = "../Datanad/subset_data"
# pattern_084 = os.path.join(path_084, "CGJ 084 -*.xlsx")
# all_files_084 = glob.glob(pattern_084)
# files_084 = [f for f in all_files_084 if not os.path.basename(f).startswith("~$")]
#
# print(f"🔍 [084] {len(files_084)} fichiers trouvés.")
#
# dfs_084 = []
#
# for i, f in enumerate(files_084):
#     try:
#         print(f"   -> reading ({i+1}/{len(files_084)}): {os.path.basename(f)}...")
#         df = pd.read_excel(f, skiprows=1)
#         dfs_084.append(df)
#     except Exception as e:
#         print(f"   ❌ Error sur {os.path.basename(f)} : {e}")
#
# if dfs_084:
#     df_084 = pd.concat(dfs_084, ignore_index=True)
#     print(f"✅ [084] Concatenation successfull : {len(df_084)} raw lines.")
#
#
#     df_map_084 = (columns={
#         "UG entrée séjour Code":            "uam_service",
#         "Nda":                              "nda",
#         "Date entrée UG entrée séjour":     "date_entree",
#         "Date sortie UG entrée séjour":     "date_sortie",
#         "Nom du patient":                   "nom",
#         "Prénom du patient":                "prenom",
#         "Date de naissance du patient":     "date_naissance",
#         "Libelle de la ville de naissance": "ville_naissance",
#         "Pays Naissance":                   "pays_naissance",
#         "Numéro de securite sociale":       "nir",
#         "Adresse du patient (rue)":         "adresse_rue",
#         "Code postal de la ville":          "adresse_cp",
#         "Code commune de la ville":         "adresse_insee",
#         "Libelle de la ville":              "adresse_ville",
#         # --- AJOUTE CES DEUX LIGNES ICI ---
#         "Mode sortie*":                     "mode_sortie",
#         "Décision urgence libellé*":        "decision_urgence",
#     }, inplace=True, errors="ignore")
#
#     print("   -> renaming done.")
# else:
#     df_084 = pd.DataFrame()
#     print("⚠️ No file CGJ 084 found.")
#
# ######################################
# # Partie 2 : CGJ 085 files
# ######################################
# path_085 = "../Datanad/subset_data"
# pattern_085 = os.path.join(path_085, "CGJ 085 -*.xlsx")
# all_files_085 = glob.glob(pattern_085)
# files_085 = [f for f in all_files_085 if not os.path.basename(f).startswith("~$")]
#
# print(f"\n🔍 [085] {len(files_085)} files found.")
#
# dfs_085 = []
# for i, f in enumerate(files_085):
#     try:
#         print(f"   -> Reading ({i+1}/{len(files_085)}): {os.path.basename(f)}...")
#         # begin at line 4
#         df = pd.read_excel(f, skiprows=3)
#         dfs_085.append(df)
#     except Exception as e:
#         print(f"   ❌ Error sur {os.path.basename(f)} : {e}")
#
# if dfs_085:
#     df_085 = pd.concat(dfs_085, ignore_index=True)
#     print(f"✅ [085] Concatenation successfull : {len(df_085)} raw lines.")
#
#     rename_map_085 = {
#         "UAM entrée venue": "uam_service",
#         "No venue": "nda",
#         "Nom usuel patient": "nom",
#         "Date de naissance": "date_naissance",
#         "Ville de naissance": "ville_naissance",
#         "Pays de naissance": "pays_naissance",
#         "Prénom usuel patient": "prenom",
#         "Date d'entrée venue": "date_entree",
#         "Numéro SS assuré": "nir",
#         "1ere ligne adresse": "adresse_rue",
#         "Code postal": "adresse_cp",
#         "Ville de résidence": "adresse_ville",
#     }
#     df_085.rename(columns=rename_map_085, inplace=True, errors="ignore")
#     print("   -> Renaming done.")
# else:
#     df_085 = pd.DataFrame()
#     print("⚠️ Not file CGJ 085 found.")
#
# ######################################
# # Partie 3 : fusion and final cleaning
# ######################################
# print("\n🔄 Fusion of the 2 sources (084 + 085)...")
# df_final = pd.concat([df_084, df_085], ignore_index=True)
#
# if df_final.empty:
#     print("❌ Empty results. Check file paths.")
# else:
#     # NDA cleaning (against doublons 12345 et 12345.0)
#     print("   -> Cleaning NDA...")
#     df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#
#     # Smart removing duplicates (keep the most complete line per NDA)
#     print("   -> removing duplicates (prioryty to the most complete lines)...")
#     df_final["count_nonnull"] = df_final.notna().sum(axis=1)
#     df_final.sort_values(by="count_nonnull", ascending=False, inplace=True)
#
#     n_avant = len(df_final)
#     df_final.drop_duplicates(subset="nda", keep="first", inplace=True)
#     n_apres = len(df_final)
#     print(f"   -> Doublons supprimés : {n_avant - n_apres} lignes.")
#
#     # Final sorting by date of entry (keep the earliest entries at the top)
#     if "date_entree" in df_final.columns:
#         print("   -> Sorting by admission date...")
#         df_final["date_entree"] = pd.to_datetime(df_final["date_entree"], errors='coerce')
#         df_final.sort_values(by="date_entree", ascending=True, inplace=True)
#
#     df_admin = df_final.drop(columns=["count_nonnull"], errors="ignore").reset_index(drop=True)
#
#
# # --- FILTRE ANNÉE 2022 ---
#     if "date_entree" in df_admin.columns:
#         print("\n📅 Filtering 2022...")
#         n_avant_filtre = len(df_admin)
#
#         # Ckecking if format datetime
#         df_admin["date_entree"] = pd.to_datetime(df_admin["date_entree"], errors='coerce')
#
#         # keeping only rows where date_entree is in 2022
#         df_admin = df_admin[df_admin["date_entree"].dt.year == 2022].copy()
#
#         n_apres_filtre = len(df_admin)
#         print(f"   -> removed lines (not 2022) : {n_avant_filtre - n_apres_filtre}")
#
#
#
#
#     print("✨ SCRIPT RUNNING = SUCCESS!!!!! ✨")
#     print(f"   Final total : {len(df_admin)} unique patients.")

In [ ]:
    # # Saving !!!!!!!!!!!!!!!!!!!!!!!! a changer quand tous les fichiers !!!!!!!!!!!!!!!!!!!!!!!!!!
    # output_file = "df_admin_subset22_pel_sa.csv"
    # print(f"\n💾 Saving under : {output_file}...")
    # df_admin.to_csv(output_file, index=False)